# Wine Quality Prediction Using a Weighted Ensemble
### A reproduction of Ma et al. (2024), *Journal of Business Research* 176:114576

**SURIANANDHAN S · 127018060 · CSE425 — Machine Learning Essentials**
Guide: Dr. Vidivelli S, AP-II · Department of CSE, School of Computing

---

**To run this: `Runtime` → `Run all`.** Nothing to upload, no Google Drive to mount,
no repository to clone. Every cell below is self-contained.

This notebook rebuilds the entire project inside Colab:

1. Fetches the 1,599-sample UCI red wine dataset (with an offline fallback).
2. Writes the project's real source files to disk — you can read them in the cells below.
3. Runs the full pipeline from the single entry point, `run_all.py`.
4. Displays the results figure and tables inline.

> Generated by `notebooks/build_colab_notebook.py` from the actual project sources.
> Do not edit this notebook by hand — edit `src/` and regenerate.

## 1. Environment

Colab already ships NumPy, pandas, SciPy, scikit-learn and Matplotlib, so there is
nothing to install for the reproduction. This cell just records which versions this
run used — worth having on screen if anyone asks whether the result is
version-dependent.

In [ ]:
import sys, platform, os

print("Python     ", sys.version.split()[0], "on", platform.system())
for name in ["numpy", "pandas", "scipy", "sklearn", "matplotlib"]:
    try:
        mod = __import__(name)
        print(f"{name:<11}", getattr(mod, "__version__", "?"))
    except ImportError:
        print(f"{name:<11} MISSING")
print("CPU cores  ", os.cpu_count(), "(Colab free tier usually gives 2)")

# The tree ensembles (Random Forest, GradientBoosting) are sensitive to the
# scikit-learn version: 1.7.2 and 1.9.x give slightly different accuracies on
# this dataset even with the seed fixed. The reported results were produced on
# 1.9.x, so say so plainly rather than let a reader assume an exact match.
import sklearn
SKL_VERIFIED = ("1.9.",)
if sklearn.__version__.startswith(SKL_VERIFIED):
    print(f"\nscikit-learn {sklearn.__version__} -- matches the version the reported "
          f"results were produced on. Numbers should agree exactly.")
else:
    print(f"\nNOTE: scikit-learn {sklearn.__version__} differs from the 1.9.x the reported "
          f"results used.\n"
          f"      This was measured, not guessed. Running the full protocol on 1.7.2 gave:\n"
          f"        SVM  61.951220  and  MLP  61.350844  -- identical to 16 digits\n"
          f"        RF   71.21 vs 71.03  |  GradBoost 67.75 vs 67.67  (+0.18, +0.09)\n"
          f"        Ensemble soft 69.17 vs 69.01  |  hard 69.97 vs 69.93\n"
          f"      The ranking order, the ensemble still trailing Random Forest, and the\n"
          f"      gap against the paper's 89.73%% are all unchanged. Only the tree\n"
          f"      ensembles drift, by under 0.2 points.\n"
          f"      To match the slides exactly:  !pip install -q scikit-learn==1.9.0  then restart.")

## 2. Rebuild the project layout

`src/config.py` derives every path from its own location, so the folders have to
match the real project exactly.

In [ ]:
from pathlib import Path

for d in ["src", "data/raw", "results/tables", "results/figures"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
!find . -maxdepth 2 -type d -not -path './.*' | sort

## 3. Get the dataset

Downloads `winequality-red.csv` from the UCI Machine Learning Repository. If the
network is unavailable, it falls back to a copy embedded in the next cell — so this
notebook still works with no internet at all.

Either way the file is verified: it must parse to **1,599 rows × 12 columns**
before the notebook continues.

In [ ]:
import base64, urllib.request, io
import pandas as pd

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
DEST = Path("data/raw/winequality-red.csv")
EXPECTED_ROWS, EXPECTED_COLS = 1599, 12

source = None
try:
    with urllib.request.urlopen(URL, timeout=30) as r:
        DEST.write_bytes(r.read())
    source = "downloaded from UCI"
except Exception as e:
    print(f"Download unavailable ({type(e).__name__}) -- using the embedded copy.")
    DEST.write_bytes(base64.b64decode(EMBEDDED_CSV_B64))
    source = "embedded fallback copy"

df = pd.read_csv(DEST, sep=";")
assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), f"unexpected shape {df.shape}"
print(f"Dataset ready ({source}): {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nQuality class counts -- note the imbalance, this is slide 9:")
print(df["quality"].value_counts().sort_index().to_string())
df.head()

<details>
<summary><b>The embedded fallback copy of the dataset</b> (click to expand — you do not
need to read this, but it must be run before the cell above)</summary>
</details>

This cell only defines a string. Colab's `Run all` executes cells top to bottom, so it
is placed here deliberately — **if you run cells by hand, run this one first.**

In [ ]:
EMBEDDED_CSV_B64 = """\
...
"""
print(f"Embedded fallback: {len(EMBEDDED_CSV_B64):,} base64 characters")

## 4. The project source

These are the real files from the project, not a simplified rewrite. Each cell writes
one module to disk and displays its source, so everything the panel is asked to believe
about the implementation is visible here.

| Module | What it does |
|---|---|
| `config.py` | Paths, the 10×10 protocol, the fixed seed, and the published benchmark numbers |
| `preprocessing.py` | Loads the CSV into X (11 features) and y (quality 3–8) |
| `base_learners.py` | SVM, Random Forest, MLP, GradientBoosting — each inside a scaling `Pipeline` |
| `ensemble.py` | Equations (1) and (4)–(9); soft and hard weighted voting; the 10×10 CV loop |
| `evaluate.py` | Confusion matrices, per-class metrics, and the reconciliation against the paper |
| `review2.py` | Review 2: class balancing, weight sharpening, learned temperature, stacking |
| `feature_importance.py` | Review 2: permutation importance, direction of effect, how many lab tests are needed |
| `run_all.py` | The single entry point |

In [ ]:
%%writefile src/config.py
"""
config.py

Single source of truth for paths, experiment settings, and the published
benchmark numbers we compare against. Previously these constants were
duplicated across evaluate.py and novelty_cluster_mine_feedback.py.
"""
from pathlib import Path

# --- Paths ----------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
RED_WINE_CSV = DATA_RAW_DIR / "winequality-red.csv"
WHITE_WINE_CSV = DATA_RAW_DIR / "winequality-white.csv"

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

# Reduced-protocol runs (run_all.py --quick) write here instead of results/, so
# a fast sanity pass can never overwrite the canonical 10-repeat numbers that
# README.md and report/methodology.md quote. Gitignored.
QUICK_OUTPUT_DIR = PROJECT_ROOT / "results" / "_quick_run"

# Review 2 (class balancing, improvements, feature importance) writes to its own
# folder so the Review-1 reproduction numbers in results/tables/ stay
# byte-identical and can always be quoted as the baseline.
REVIEW2_DIR = RESULTS_DIR / "review2"
REVIEW2_TABLES_DIR = REVIEW2_DIR / "tables"
REVIEW2_FIGURES_DIR = REVIEW2_DIR / "figures"
QUICK_REVIEW2_DIR = QUICK_OUTPUT_DIR / "review2"

# --- Dataset --------------------------------------------------------------
FEATURE_COLUMNS = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]
TARGET_COLUMN = "quality"

# --- Experiment protocol --------------------------------------------------
RANDOM_STATE = 42
N_SPLITS = 10       # 10-fold cross-validation
N_REPEATS = 10      # x 10 runs  => 100 experiments, as in Ma et al. (2024) Sec. 3.4
QUICK_N_REPEATS = 2  # used by run_all.py --quick for a fast sanity pass

# --- Published benchmark numbers (context only, not our results) ----------
# Classes 0,1,2,9,10 are always empty in this dataset and are omitted, exactly
# as Ma et al. (2024) omit them from their Tables 4-9.
BENCHMARK_CLASSES = [3, 4, 5, 6, 7, 8]
CORTEZ_2009_PER_CLASS_PRECISION = [0.00, 20.00, 67.50, 57.70, 58.60, 0.00]
MA_2024_PER_CLASS_PRECISION = [33.33, 63.16, 84.80, 88.06, 77.08, 50.00]
MA_2024_OVERALL_ACCURACY = 89.73

# Ma et al. (2024) Table 9 -- the ensemble confusion matrix, transcribed by eye
# from the printed table. Rows = prognostic (predicted) class, columns = actual
# class, both ordered [3, 4, 5, 6, 7, 8].
#
# CAVEAT: transcribed visually from the PDF and may contain reading errors.
# evaluate.paper_table9_reconciliation() uses this only to sanity-check the
# paper's reported figures against its own printed table; any conclusion drawn
# from it is stated as "could not reconcile", never as "the paper is wrong".
MA_2024_TABLE9_CONFUSION = [
    [1, 1, 1, 0, 0, 0],
    [0, 12, 6, 4, 0, 0],
    [2, 6, 173, 7, 11, 0],
    [0, 0, 14, 177, 11, 0],
    [0, 0, 10, 12, 37, 2],
    [0, 0, 0, 0, 1, 3],
]

# Range of overall accuracy commonly reported in the literature for the 6-class
# UCI red wine task, used for context in the report.
#
# CAVEAT: this band is an informal impression from the literature survey, NOT a
# figure compiled from a citation table. It is quoted as "typically reported"
# rather than as an established result, and tightening it with explicit
# citations is listed as future work. Do not present it as a sourced statistic.
LITERATURE_ACCURACY_RANGE_PCT = (62.0, 72.0)

# Ma et al. (2024) Figs. 4-5: the physicochemical attributes the paper singles
# out as most important. Review 2's feature-importance analysis is checked
# against this list.
PAPER_KEY_ATTRIBUTES = ["alcohol", "sulphates", "volatile acidity", "citric acid"]

# --- Review 2 settings ----------------------------------------------------
# PRIMARY METRIC, fixed before any Review-2 experiment was run.
# Accuracy rewards predicting the two majority classes (5 and 6 are 82.5% of the
# data) and is precisely what class balancing trades away to rescue the rare
# classes. Macro-F1 averages F1 over all six quality classes with equal weight,
# so it is the metric class balancing is meant to improve. Accuracy is still
# reported everywhere, for comparability with the paper.
REVIEW2_PRIMARY_METRIC = "macro_f1"

# SMOTE neighbours. The rarest class (3) has 10 samples, so a 10-fold training
# split holds 9 and a stacking inner split about 7 -- both enough for k = 5.
SMOTE_K_NEIGHBORS = 5

# Temperatures for the weight-sharpening test of the Review-1 diagnosis.
# T = 1 is the paper's Eqs. (5) and (9) exactly. Capped at 100 so the powers
# stay finite in float64.
WEIGHT_TEMPERATURES = [1, 2, 5, 10, 25, 100]

# Feature-subset sizes for the "how many lab tests do you need?" experiment.
FEATURE_SUBSET_SIZES = [1, 2, 3, 4, 6, 8, 11]

# Permutations per feature per fold for permutation importance.
PERMUTATION_REPEATS = 20

# Ma et al. (2024) Tables 4-7: overall accuracy of each individual base learner.
MA_2024_BASE_LEARNER_ACCURACY = {
    "SVM": 67.30,
    "RNN": 61.43,
    "ELM": 55.77,
    "RF": 80.71,
}

# Ma et al. (2024) Table 8: the final "weight-sum" w_i per base learner,
# i.e. Eq. (4)'s 0.5 * (w_accuracy + w_diversity). Note how close these are to
# a uniform 0.25 -- see evaluate.paper_internal_consistency_check().
MA_2024_TABLE8_WEIGHT_SUM = {
    "SVM": 0.2516,
    "RNN": 0.2286,
    "ELM": 0.2168,
    "RF": 0.3030,
}

In [ ]:
%%writefile src/preprocessing.py
"""
preprocessing.py

Data loading and preprocessing for the UCI red wine quality dataset, matching
the preprocessing step in Ma et al. (2024) Fig. 2: raw data -> min-max
normalization -> downstream learners. No rows are dropped here; class 3/4/8
scarcity is handled explicitly downstream (stratified CV, balanced class
weights), not by discarding data.

WHERE THE SCALING HAPPENS: the min-max step now lives inside each learner's
Pipeline (see base_learners._pipe), so it is fit on the training fold only.
Use `load_red_wine_xy()` -- which returns RAW, unscaled X -- for anything that
feeds cross-validation. `load_normalized_red_wine()` is retained for standalone
inspection and for the optional novelty module, which fits unsupervised models
on the whole dataset at once (no train/test split, so no leakage concern).
"""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from config import FEATURE_COLUMNS, RED_WINE_CSV, TARGET_COLUMN

DEFAULT_RED_WINE_PATH = RED_WINE_CSV


def load_red_wine(path=DEFAULT_RED_WINE_PATH):
    """Load the red wine CSV (semicolon-delimited, as distributed by UCI)."""
    return pd.read_csv(path, sep=";")


def get_features_target(df, feature_columns=None):
    feature_columns = feature_columns or FEATURE_COLUMNS
    X = df[feature_columns].to_numpy(dtype=float)
    y = df[TARGET_COLUMN].to_numpy(dtype=int)
    return X, y


def minmax_normalize(X):
    """Min-max scale each feature to [0, 1] (paper's own preprocessing step)."""
    scaler = MinMaxScaler()
    X_norm = scaler.fit_transform(X)
    return X_norm, scaler


def train_test_split_data(X, y, test_size=0.2, random_state=42):
    """Stratified hold-out split, used only for the illustrative single-split
    metrics in base_learners.py. The paper's headline numbers (and ours, in
    ensemble.py) come from repeated k-fold CV, not this split."""
    return train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )


def load_red_wine_xy(path=DEFAULT_RED_WINE_PATH, feature_columns=None):
    """Load the dataset and return RAW (unscaled) features.

    This is the loader every cross-validated experiment should use: scaling is
    applied inside each learner's Pipeline, fit on the training fold only.

    Returns (X_raw, y, feature_columns, df).
    """
    df = load_red_wine(path)
    feature_columns = feature_columns or FEATURE_COLUMNS
    X, y = get_features_target(df, feature_columns)
    return X, y, feature_columns, df


def load_normalized_red_wine(path=DEFAULT_RED_WINE_PATH, feature_columns=None):
    """Convenience wrapper: load -> extract features/target -> min-max normalize.

    NOTE: scales using the whole dataset. Safe for unsupervised analysis over
    all rows at once (the optional novelty module), but do NOT feed the result
    into cross-validation -- use `load_red_wine_xy()` for that instead.

    Returns (X_norm, y, feature_columns, df).
    """
    df = load_red_wine(path)
    feature_columns = feature_columns or FEATURE_COLUMNS
    X, y = get_features_target(df, feature_columns)
    X_norm, _ = minmax_normalize(X)
    return X_norm, y, feature_columns, df


if __name__ == "__main__":
    df = load_red_wine()
    print("shape:", df.shape)
    print(df.head())
    print("\nquality value_counts:")
    print(df[TARGET_COLUMN].value_counts().sort_index())

    X, y = get_features_target(df)
    X_norm, _ = minmax_normalize(X)
    print("\nnormalized feature range check (min, max) per column:")
    print(np.round(X_norm.min(axis=0), 3))
    print(np.round(X_norm.max(axis=0), 3))

In [ ]:
%%writefile src/base_learners.py
"""
base_learners.py

Individual base-learner definitions and 10-fold cross-validation, standing in
for the four base learners of Ma et al. (2024): SVM, RNN, ELM, and Random
Forest.

SUBSTITUTIONS (classical-ML-only university syllabus -- no deep learning
beyond a standard backprop MLP, no non-backprop training paradigms):

  - RNN (recurrent neural network; a deep-learning sequence model) is replaced
    with GradientBoostingClassifier. Both sit outside the "single flat model"
    mold of the original ensemble: gradient boosting fits each new tree to the
    residual errors of the trees before it, which is a classical-ML way of
    letting the model refine itself round over round -- the closest in-syllabus
    analogue to the RNN's iterative refinement, without backprop-through-time
    or exploding/vanishing-gradient machinery. (The project brief permits
    "AdaBoostClassifier / GradientBoostingClassifier"; we use gradient boosting
    because AdaBoost's default depth-1 stumps perform poorly on this 6-class
    imbalanced problem.)
  - ELM (extreme learning machine; random, fixed hidden-layer weights with a
    closed-form linear output layer, i.e. explicitly *not* trained with
    backprop) is replaced with MLPClassifier. An MLP is the closest classical
    analogue that is still a neural network, but -- unlike ELM -- all of its
    weights (hidden and output) are learned via standard backpropagation,
    which is what the syllabus covers.
  - SVM and Random Forest are kept as-is: both are already classical ML and
    directly available in scikit-learn.

PREPROCESSING NOTE (important): every learner is wrapped in a Pipeline whose
first step is MinMaxScaler. This keeps the paper's min-max normalization step
(Fig. 2) while ensuring the scaler is fit on the *training fold only* during
cross-validation. Scaling the full dataset once before CV -- as an earlier
version of this project did -- leaks test-fold information into the scaling
constants and inflates the reported scores.

CLASS IMBALANCE: quality classes 3 and 8 have only 10 and 18 samples out of
1599. SVM and RF therefore use balanced class weights so the minority classes
contribute to the loss instead of being ignored outright. This is a modelling
choice we make explicitly; the paper does not state any imbalance handling.
"""
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

from config import N_SPLITS, RANDOM_STATE


def _pipe(estimator):
    """Min-max scaling fit inside each CV fold, then the estimator."""
    return Pipeline([("scaler", MinMaxScaler()), ("clf", estimator)])


def get_base_learners(random_state=RANDOM_STATE):
    """Return the four base learners, keyed by name, each as a scaling pipeline.

    Keys "MLP" and "GradBoost" stand in for the paper's "ELM" and "RNN"
    respectively -- see module docstring for why.
    """
    return {
        # The soft-vote ensemble (Eq. 1) needs probability estimates, which an
        # SVM does not produce natively. SVC(probability=True) would supply them,
        # but it refits an internal 5-fold Platt calibration whose result can
        # disagree with the model's own decision_function. Wrapping in
        # CalibratedClassifierCV makes that calibration step explicit and keeps
        # it inside the CV fold. (Both routes use Platt scaling; this is a
        # transparency choice, not a deprecation -- SVC(probability=True) is
        # still perfectly valid.)
        "SVM": _pipe(
            CalibratedClassifierCV(
                SVC(
                    kernel="rbf",
                    C=10.0,
                    gamma="scale",
                    class_weight="balanced",
                    random_state=random_state,
                ),
                ensemble=False,
            )
        ),
        "RF": _pipe(
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced_subsample",
                random_state=random_state,
            )
        ),
        "MLP": _pipe(
            MLPClassifier(
                hidden_layer_sizes=(64, 32),
                max_iter=2000,
                # No early_stopping: it would carve a 10% validation slice out
                # of a training set containing as few as 9 class-3 samples.
                random_state=random_state,
            )
        ),
        "GradBoost": _pipe(
            GradientBoostingClassifier(
                n_estimators=200,
                random_state=random_state,
            )
        ),
    }


def cross_val_predictions(X, y, n_splits=N_SPLITS, random_state=RANDOM_STATE):
    """Out-of-fold predictions for each base learner under one shared
    stratified k-fold split (so every learner is scored on the same folds)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    learners = get_base_learners(random_state)
    return {
        name: cross_val_predict(model, X, y, cv=skf, n_jobs=-1)
        for name, model in learners.items()
    }


def cross_val_accuracy_scores(X, y, n_splits=N_SPLITS, random_state=RANDOM_STATE):
    """Per-fold accuracy scores for each base learner (for the mean/std
    summary reported alongside the pooled confusion-matrix metrics)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    learners = get_base_learners(random_state)
    return {
        name: cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
        for name, model in learners.items()
    }


if __name__ == "__main__":
    from preprocessing import load_red_wine_xy

    X, y, feature_columns, _ = load_red_wine_xy()
    scores = cross_val_accuracy_scores(X, y)
    for name, folds in scores.items():
        print(f"{name}: mean accuracy = {folds.mean():.4f} (+/- {folds.std():.4f}) over {len(folds)} folds")

In [ ]:
%%writefile src/ensemble.py
"""
ensemble.py

The weighted ensemble of Ma et al. (2024), Section 3.3.3 / Eqs. (1)-(9):

  y_hat   = sum_i w_i * y_hat_i                                        (1)
  w_i     = 0.5 * (w_accuracy_i + w_diversity_i)                       (4)
  w_accuracy_i  = (eps_i_CV)^-1 / sum_j (eps_j_CV)^-1                  (5)
  eps_i_CV      = mean 0/1 CV error of learner i over all folds        (6)
  e_i           = per-sample 0/1 error vector of learner i (Eq. 7)
  D_i           = sum_{j != i} || e_i - e_j ||                        (8)
  w_diversity_i = D_i / sum_j D_j                                      (9)

HOW Eq. (1) IS REALIZED -- and why it matters
---------------------------------------------
Eq. (1) is a weighted *sum* of each learner's output, y_hat = sum_i w_i y_hat_i.
There are two ways to read y_hat_i for a classifier, and they give materially
different results:

  SOFT vote (used for our headline result): y_hat_i is the per-class
    probability vector P_i from predict_proba. The ensemble output is the
    weighted sum of those vectors, P = sum_i w_i * P_i, and the prediction is
    argmax(P). This is the literal reading of Eq. (1) -- an actual weighted sum
    of scores -- and it lets a confident strong learner outweigh several
    unconfident weak ones.

  HARD vote (kept for comparison): y_hat_i is the one-hot encoded predicted
    label, so the "sum" degenerates into weighted majority voting. On this
    imbalanced 6-class problem it lets three weaker learners jointly outvote
    the strongest one, and measurably underperforms the soft vote.

Both are computed and reported, since the difference between them is a genuine
finding about how the paper's formula should be interpreted.

Evaluation protocol: 10-fold CV repeated 10 times (100 total fold-runs),
matching the paper's "10 runs of 10-fold cross-validation" (Section 3.4).
"""
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from base_learners import get_base_learners
from config import N_REPEATS, N_SPLITS


def _error_vector(y_true, y_pred):
    """g(y_hat, y) from Eq. (3): 0 if correct, 1 if wrong."""
    return (np.asarray(y_true) != np.asarray(y_pred)).astype(float)


def compute_weights(error_vectors):
    """error_vectors: dict[name] -> 0/1 error array of length N (Eq. 7).

    Returns (w, w_accuracy, w_diversity, cv_errors), each a dict[name] -> float,
    following Eqs. (4), (5), (8), (9).
    """
    names = list(error_vectors.keys())
    cv_errors = {name: float(error_vectors[name].mean()) for name in names}  # Eq. (6)

    # Eq. (5): accuracy-based weight. A learner with zero CV error would
    # divide by zero; fall back to a large-but-finite inverse error instead
    # (equivalent to treating it as "as good as effectively perfect").
    inv_err = {name: (1.0 / cv_errors[name] if cv_errors[name] > 0 else 1e6) for name in names}
    total_inv = sum(inv_err.values())
    w_accuracy = {name: inv_err[name] / total_inv for name in names}

    # Eq. (8): diversity = sum of Euclidean distances to every other learner's
    # error vector.
    D = {}
    for name in names:
        D[name] = sum(
            np.linalg.norm(error_vectors[name] - error_vectors[other])
            for other in names
            if other != name
        )
    total_D = sum(D.values())
    w_diversity = {name: D[name] / total_D for name in names}  # Eq. (9)

    w = {name: 0.5 * (w_accuracy[name] + w_diversity[name]) for name in names}  # Eq. (4)
    return w, w_accuracy, w_diversity, cv_errors


def weight_concentration(weights):
    """How far the paper's weights depart from a plain uniform average.

    This matters for interpreting the reproduction. Eqs. (5) and (9) normalize
    inverse-errors and pairwise error distances, and both are only weakly
    discriminative when the base learners have broadly similar error rates: a
    learner with 29% error and one with 37% error receive weights of roughly
    0.27 and 0.21. The paper's own Table 8 shows the same near-uniform pattern
    (0.2516 / 0.2286 / 0.2168 / 0.3030 against a uniform 0.25).

    With weights this flat, the ensemble is close to an unweighted average of
    its members, and an average of one strong learner and three weaker ones
    lands *between* them rather than above the strongest. Reported as a
    diagnostic so this can be checked rather than asserted.

    Returns dict with the uniform baseline, max/min weight, spread, and the
    max absolute deviation from uniform.
    """
    values = np.array(list(weights.values()), dtype=float)
    uniform = 1.0 / len(values)
    return {
        "n_learners": len(values),
        "uniform_weight": uniform,
        "max_weight": float(values.max()),
        "min_weight": float(values.min()),
        "weight_spread": float(values.max() - values.min()),
        "max_abs_deviation_from_uniform": float(np.abs(values - uniform).max()),
    }


def weighted_vote(preds, weights, classes):
    """HARD-vote realization of Eq. (1): each learner one-hot votes its predicted
    label with weight w_i. Kept for comparison against the soft vote."""
    names = list(preds.keys())
    n_samples = len(next(iter(preds.values())))
    class_index = {c: i for i, c in enumerate(classes)}
    votes = np.zeros((n_samples, len(classes)))
    for name in names:
        w = weights[name]
        idx = np.array([class_index[p] for p in preds[name]])
        votes[np.arange(n_samples), idx] += w
    return np.array([classes[i] for i in votes.argmax(axis=1)])


def weighted_soft_vote(probas, weights, classes):
    """SOFT-vote realization of Eq. (1) -- the literal weighted sum of scores.

    probas: dict[name] -> (n_samples, n_classes) predict_proba matrix, with
    columns ordered to match `classes`. Computes P = sum_i w_i * P_i and
    returns argmax(P) as a class label.
    """
    names = list(probas.keys())
    weighted_sum = np.zeros_like(next(iter(probas.values())), dtype=float)
    for name in names:
        weighted_sum += weights[name] * probas[name]
    classes_arr = np.asarray(classes)
    return classes_arr[weighted_sum.argmax(axis=1)]


def _align_proba_columns(proba, model_classes, classes):
    """Reorder/pad a predict_proba matrix so its columns match `classes`.

    A learner trained on a fold that happens to contain no samples of some rare
    class will return fewer columns; those classes get probability 0.
    """
    model_classes = list(model_classes)
    if model_classes == list(classes):
        return proba
    aligned = np.zeros((proba.shape[0], len(classes)))
    for j, cls in enumerate(model_classes):
        aligned[:, classes.index(cls)] = proba[:, j]
    return aligned


def run_repeated_cv(X, y, n_splits=N_SPLITS, n_repeats=N_REPEATS, base_seed=0):
    """Run n_repeats independent n_splits-fold stratified CVs (matching the
    paper's 10-fold CV x 10 repeats = 100 total fold-runs).

    Within each repeat, every base learner is scored on the *same* fold split
    (required for D_i in Eq. 8 to compare error vectors sample-for-sample),
    the paper's weights are computed from that repeat's out-of-fold errors,
    and the weighted-vote ensemble prediction is formed for that repeat.

    Returns a dict with:
      - "classes": sorted list of class labels
      - "per_repeat": list of per-repeat weight/accuracy records
      - "pooled_true": concatenation of y across all repeats
      - "pooled_preds": dict[learner] -> concatenation of out-of-fold preds
      - "pooled_ensemble": concatenation of ensemble preds across all repeats
      - "weights_df_records": flat list of dicts, one per repeat, for a
        weights-over-repeats table
    """
    classes = sorted(np.unique(y).tolist())
    learners = get_base_learners()

    per_repeat = []
    weights_df_records = []
    pooled_true = []
    pooled_preds = {name: [] for name in learners}
    pooled_ensemble = []       # soft vote -- the headline ensemble
    pooled_ensemble_hard = []  # hard vote -- kept for comparison

    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=base_seed + r)

        fold_preds = {}
        fold_probas = {}
        for name, model in learners.items():
            proba = cross_val_predict(model, X, y, cv=skf, n_jobs=-1, method="predict_proba")
            fold_probas[name] = _align_proba_columns(proba, classes, classes)
            # Derive the hard label from the same probabilities, so the two
            # voting schemes are compared on identical fitted models.
            fold_preds[name] = np.asarray(classes)[proba.argmax(axis=1)]

        error_vectors = {name: _error_vector(y, fold_preds[name]) for name in learners}
        w, w_acc, w_div, cv_err = compute_weights(error_vectors)

        ensemble_pred = weighted_soft_vote(fold_probas, w, classes)
        ensemble_pred_hard = weighted_vote(fold_preds, w, classes)

        accuracy = {name: float((fold_preds[name] == y).mean()) for name in learners}
        accuracy["Ensemble"] = float((ensemble_pred == y).mean())
        accuracy["Ensemble_hard_vote"] = float((ensemble_pred_hard == y).mean())

        per_repeat.append(
            {
                "repeat": r,
                "weights": w,
                "weights_accuracy": w_acc,
                "weights_diversity": w_div,
                "cv_error": cv_err,
                "accuracy": accuracy,
            }
        )
        weights_df_records.append({"repeat": r, **{f"w_{name}": w[name] for name in learners}})

        pooled_true.append(y)
        for name in learners:
            pooled_preds[name].append(fold_preds[name])
        pooled_ensemble.append(ensemble_pred)
        pooled_ensemble_hard.append(ensemble_pred_hard)

    pooled_true = np.concatenate(pooled_true)
    pooled_preds = {name: np.concatenate(v) for name, v in pooled_preds.items()}
    pooled_ensemble = np.concatenate(pooled_ensemble)
    pooled_ensemble_hard = np.concatenate(pooled_ensemble_hard)

    return {
        "classes": classes,
        "per_repeat": per_repeat,
        "weights_df_records": weights_df_records,
        "pooled_true": pooled_true,
        "pooled_preds": pooled_preds,
        "pooled_ensemble": pooled_ensemble,
        "pooled_ensemble_hard": pooled_ensemble_hard,
    }


if __name__ == "__main__":
    from preprocessing import load_red_wine_xy

    X, y, feature_columns, _ = load_red_wine_xy()
    results = run_repeated_cv(X, y, n_repeats=2)  # quick smoke test, 2 repeats only

    for name in list(results["pooled_preds"].keys()):
        acc = (results["pooled_preds"][name] == results["pooled_true"]).mean()
        print(f"{name}: pooled accuracy over {len(results['per_repeat'])} repeats = {acc:.4f}")
    soft_acc = (results["pooled_ensemble"] == results["pooled_true"]).mean()
    hard_acc = (results["pooled_ensemble_hard"] == results["pooled_true"]).mean()
    print(f"Ensemble (soft vote): pooled accuracy = {soft_acc:.4f}")
    print(f"Ensemble (hard vote): pooled accuracy = {hard_acc:.4f}")
    print("\nWeights, repeat 0:", results["per_repeat"][0]["weights"])

In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py

Confusion matrices and per-class metrics for each base learner and the final
ensemble, plus a benchmark comparison against the published numbers from
Cortez et al. (2009) and Ma et al. (2024).

A note on "precision" terminology
----------------------------------
Ma et al. (2024) label their per-class metric "precision," but their own
worked example (Eq. 14 in the paper: 137 / (137+1+2+58+6) = 67.16% for class
5) divides the diagonal count by the *total number of samples actually
belonging to that class* -- i.e. TP / (TP + FN). That is the standard
definition of recall / sensitivity, not precision (TP / (TP + FP)). To keep
our benchmark comparison numerically comparable to the numbers Ma et al. and
Cortez et al. published, we reproduce their metric under their definition,
but we label it explicitly as "precision (paper definition = recall)" and
also report conventional precision/recall/specificity separately so the two
are never conflated.
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score

from config import (
    BENCHMARK_CLASSES,
    CORTEZ_2009_PER_CLASS_PRECISION,
    FIGURES_DIR,
    LITERATURE_ACCURACY_RANGE_PCT,
    MA_2024_BASE_LEARNER_ACCURACY,
    MA_2024_OVERALL_ACCURACY,
    MA_2024_PER_CLASS_PRECISION,
    MA_2024_TABLE8_WEIGHT_SUM,
    MA_2024_TABLE9_CONFUSION,
    TABLES_DIR,
)


def specificity_per_class(cm):
    """One-vs-rest specificity = TN / (TN + FP) for each class, from a
    standard sklearn confusion matrix (rows=true, columns=predicted)."""
    n = cm.shape[0]
    total = cm.sum()
    spec = np.zeros(n)
    for i in range(n):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        spec[i] = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return spec


def per_class_metrics(y_true, y_pred, classes):
    """Per-class precision, recall/sensitivity, specificity, and the paper's
    "precision" (numerically identical to recall -- see module docstring)."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    precision = precision_score(y_true, y_pred, labels=classes, average=None, zero_division=0)
    recall = recall_score(y_true, y_pred, labels=classes, average=None, zero_division=0)
    specificity = specificity_per_class(cm)

    return pd.DataFrame(
        {
            "class": classes,
            "n_actual": cm.sum(axis=1),
            "precision": precision * 100,
            "sensitivity_recall": recall * 100,
            "specificity": specificity * 100,
            "precision_paper_definition_pct": recall * 100,  # see module docstring
        }
    )


def overall_accuracy(y_true, y_pred):
    return float((np.asarray(y_true) == np.asarray(y_pred)).mean()) * 100


def confusion_matrix_df(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    return pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])


def save_model_evaluation(name, y_true, y_pred, classes, tables_dir=TABLES_DIR):
    tables_dir.mkdir(parents=True, exist_ok=True)
    cm_df = confusion_matrix_df(y_true, y_pred, classes)
    cm_df.to_csv(tables_dir / f"confusion_matrix_{name}.csv")

    metrics_df = per_class_metrics(y_true, y_pred, classes)
    metrics_df.attrs["overall_accuracy_pct"] = overall_accuracy(y_true, y_pred)
    metrics_df.to_csv(tables_dir / f"metrics_{name}.csv", index=False)

    return cm_df, metrics_df


def build_benchmark_comparison(our_metrics_df, our_overall_accuracy, tables_dir=TABLES_DIR):
    """Build the reproduction-vs-published comparison table (per class +
    overall), matching the classes/order Ma et al. (2024) report in Table 9."""
    our_by_class = our_metrics_df.set_index("class")["precision_paper_definition_pct"]
    our_values = [our_by_class.get(c, np.nan) for c in BENCHMARK_CLASSES]

    comparison = pd.DataFrame(
        {
            "class": BENCHMARK_CLASSES + ["overall"],
            "cortez_2009_pct": CORTEZ_2009_PER_CLASS_PRECISION + [np.nan],
            "ma_2024_pct": MA_2024_PER_CLASS_PRECISION + [MA_2024_OVERALL_ACCURACY],
            "our_reproduction_pct": list(our_values) + [our_overall_accuracy],
        }
    )
    tables_dir.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(tables_dir / "benchmark_comparison.csv", index=False)
    return comparison


def plot_benchmark_comparison(comparison_df, figures_dir=FIGURES_DIR, filename="accuracy_comparison.png"):
    """Grouped bar chart: our reproduction vs. Ma et al. (2024) vs. Cortez et
    al. (2009), per class and overall."""
    figures_dir.mkdir(parents=True, exist_ok=True)
    labels = [str(c) for c in comparison_df["class"]]
    x = np.arange(len(labels))
    width = 0.26

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width, comparison_df["cortez_2009_pct"], width, label="Cortez et al. (2009)")
    ax.bar(x, comparison_df["ma_2024_pct"], width, label="Ma et al. (2024) ensemble")
    ax.bar(x + width, comparison_df["our_reproduction_pct"], width, label="Our reproduction")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Quality class (last bar group = overall accuracy)")
    ax.set_ylabel("Precision / accuracy (%)  [paper's precision = per-class recall]")
    ax.set_title("Reproduction vs. published benchmarks")
    ax.legend()
    ax.set_ylim(0, 100)
    fig.tight_layout()
    fig.savefig(figures_dir / filename, dpi=150)
    plt.close(fig)


def paper_table9_reconciliation(tables_dir=TABLES_DIR):
    """Check Ma et al. (2024)'s reported headline figures against their own Table 9.

    The paper states 89.73% overall accuracy from "10 runs of 10-fold
    cross-validation" (100 experiments over all 1599 samples). We recompute
    directly from the confusion matrix they print, to understand the gap
    between their result and ours.

    IMPORTANT CAVEAT: the matrix in config.MA_2024_TABLE9_CONFUSION was
    transcribed by eye from the printed PDF table and may contain reading
    errors. Everything below is therefore reported as "we could not reconcile",
    not as a claim that the paper is wrong.
    """
    cm = np.array(MA_2024_TABLE9_CONFUSION, dtype=int)
    total_n = int(cm.sum())
    diagonal = int(np.trace(cm))
    implied_accuracy = 100.0 * diagonal / total_n

    # Their per-class percentages: does TP/column-total or TP/row-total reproduce them?
    col_totals = cm.sum(axis=0)
    row_totals = cm.sum(axis=1)
    tp = np.diag(cm)
    col_wise = np.where(col_totals > 0, 100.0 * tp / np.maximum(col_totals, 1), np.nan)
    row_wise = np.where(row_totals > 0, 100.0 * tp / np.maximum(row_totals, 1), np.nan)

    per_class = pd.DataFrame(
        {
            "class": BENCHMARK_CLASSES,
            "paper_reported_pct": MA_2024_PER_CLASS_PRECISION,
            "recomputed_column_wise_pct": np.round(col_wise, 2),
            "recomputed_row_wise_pct": np.round(row_wise, 2),
        }
    )
    per_class["column_wise_matches"] = np.isclose(
        per_class["paper_reported_pct"], per_class["recomputed_column_wise_pct"], atol=0.6
    )

    lo, hi = LITERATURE_ACCURACY_RANGE_PCT
    summary = pd.DataFrame(
        [
            ("implied_test_set_size", total_n),
            ("full_dataset_size", 1599),
            ("implied_fraction_of_dataset_pct", round(100.0 * total_n / 1599, 2)),
            ("correct_predictions_on_diagonal", diagonal),
            ("implied_overall_accuracy_pct", round(implied_accuracy, 2)),
            ("paper_reported_overall_accuracy_pct", MA_2024_OVERALL_ACCURACY),
            ("discrepancy_pct_points", round(MA_2024_OVERALL_ACCURACY - implied_accuracy, 2)),
            ("n_per_class_matching_column_wise", int(per_class["column_wise_matches"].sum())),
            ("literature_accuracy_range_pct", f"{lo}-{hi}"),
        ],
        columns=["quantity", "value"],
    )

    tables_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(tables_dir / "paper_table9_reconciliation.csv", index=False)
    per_class.to_csv(tables_dir / "paper_table9_per_class_check.csv", index=False)
    return summary, per_class


def paper_internal_consistency_check(our_ensemble_accuracy=None, tables_dir=TABLES_DIR):
    """Check the paper's ensemble result against its own base-learner numbers.

    Ma et al. report, in their own tables: base-learner accuracies of 67.30
    (SVM), 61.43 (RNN), 55.77 (ELM), 80.71 (RF) in Tables 4-7; near-uniform
    final weights of 0.2516 / 0.2286 / 0.2168 / 0.3030 in Table 8; and an
    ensemble accuracy of 89.73%.

    Applying their Table 8 weights to their Table 4-7 accuracies gives a
    weighted-average accuracy in the high 60s. A voting ensemble is not simply
    the weighted average of its members' accuracies -- decorrelated errors let
    a good ensemble exceed that baseline -- so this is an indicative reference
    point, not a proof. But exceeding it by ~22 percentage points, while using
    weights that are within 0.05 of uniform, is a large gap to explain.

    We report the number and let the reader judge. Our own ensemble result
    lands close to this weighted-average baseline, which is what one would
    expect from near-uniform weights.
    """
    weights = MA_2024_TABLE8_WEIGHT_SUM
    accuracies = MA_2024_BASE_LEARNER_ACCURACY
    weighted_avg = sum(weights[k] * accuracies[k] for k in weights)
    best_base = max(accuracies.values())
    uniform = 1.0 / len(weights)
    max_dev = max(abs(w - uniform) for w in weights.values())

    rows = [
        ("weight_sum_total", round(sum(weights.values()), 4)),
        ("max_abs_deviation_from_uniform_weight", round(max_dev, 4)),
        ("best_single_base_learner_pct", best_base),
        ("weighted_average_of_base_accuracies_pct", round(weighted_avg, 2)),
        ("paper_reported_ensemble_pct", MA_2024_OVERALL_ACCURACY),
        ("gap_vs_weighted_average_pct_points", round(MA_2024_OVERALL_ACCURACY - weighted_avg, 2)),
        ("gap_vs_best_base_learner_pct_points", round(MA_2024_OVERALL_ACCURACY - best_base, 2)),
    ]
    if our_ensemble_accuracy is not None:
        rows.append(("our_ensemble_pct", round(our_ensemble_accuracy, 2)))
        rows.append(
            ("our_gap_vs_their_weighted_average_pct_points", round(our_ensemble_accuracy - weighted_avg, 2))
        )

    check = pd.DataFrame(rows, columns=["quantity", "value"])
    tables_dir.mkdir(parents=True, exist_ok=True)
    check.to_csv(tables_dir / "paper_internal_consistency_check.csv", index=False)
    return check


def build_accuracy_summary(pooled_true, pooled_preds, pooled_ensemble, pooled_ensemble_hard=None,
                           tables_dir=TABLES_DIR):
    """One row per model, sorted best-first, so the ensemble-vs-base-learner
    comparison (the paper's central claim) can be checked at a glance."""
    rows = [
        {"model": name, "overall_accuracy_pct": overall_accuracy(pooled_true, preds)}
        for name, preds in pooled_preds.items()
    ]
    rows.append(
        {"model": "Ensemble (soft vote)", "overall_accuracy_pct": overall_accuracy(pooled_true, pooled_ensemble)}
    )
    if pooled_ensemble_hard is not None:
        rows.append(
            {
                "model": "Ensemble (hard vote)",
                "overall_accuracy_pct": overall_accuracy(pooled_true, pooled_ensemble_hard),
            }
        )

    summary = pd.DataFrame(rows).sort_values("overall_accuracy_pct", ascending=False).reset_index(drop=True)
    summary["rank"] = summary.index + 1
    tables_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(tables_dir / "model_accuracy_summary.csv", index=False)
    return summary


def evaluate_all(pooled_true, pooled_preds, pooled_ensemble, classes, pooled_ensemble_hard=None,
                 tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR):
    """Run save_model_evaluation for every base learner plus the ensemble,
    then build and plot the benchmark comparison. Returns dict[name] -> (cm_df, metrics_df).

    `tables_dir` / `figures_dir` default to the project's results/ folders. Pass a
    scratch directory when running an exploratory experiment (e.g. the walkthrough
    notebook, which uses fewer CV repeats) so that partial results cannot silently
    overwrite the canonical outputs quoted in README.md and report/methodology.md.
    """
    all_results = {}
    for name, preds in pooled_preds.items():
        all_results[name] = save_model_evaluation(name, pooled_true, preds, classes, tables_dir=tables_dir)

    ensemble_cm, ensemble_metrics = save_model_evaluation(
        "Ensemble", pooled_true, pooled_ensemble, classes, tables_dir=tables_dir
    )
    all_results["Ensemble"] = (ensemble_cm, ensemble_metrics)

    if pooled_ensemble_hard is not None:
        all_results["Ensemble_hard_vote"] = save_model_evaluation(
            "Ensemble_hard_vote", pooled_true, pooled_ensemble_hard, classes, tables_dir=tables_dir
        )

    ensemble_overall_acc = overall_accuracy(pooled_true, pooled_ensemble)
    comparison = build_benchmark_comparison(ensemble_metrics, ensemble_overall_acc, tables_dir=tables_dir)
    plot_benchmark_comparison(comparison, figures_dir=figures_dir)
    build_accuracy_summary(
        pooled_true, pooled_preds, pooled_ensemble, pooled_ensemble_hard, tables_dir=tables_dir
    )
    paper_table9_reconciliation(tables_dir=tables_dir)
    paper_internal_consistency_check(our_ensemble_accuracy=ensemble_overall_acc, tables_dir=tables_dir)

    return all_results, comparison


if __name__ == "__main__":
    from ensemble import run_repeated_cv
    from preprocessing import load_red_wine_xy

    X, y, feature_columns, _ = load_red_wine_xy()
    results = run_repeated_cv(X, y, n_repeats=2)
    _, comparison = evaluate_all(
        results["pooled_true"],
        results["pooled_preds"],
        results["pooled_ensemble"],
        results["classes"],
        pooled_ensemble_hard=results["pooled_ensemble_hard"],
    )
    print(comparison)
    print()
    summary, per_class = paper_table9_reconciliation()
    print(summary.to_string(index=False))

In [ ]:
%%writefile src/review2.py
"""
review2.py

Review 2 -- the changes requested by the guide at Review 1:

  A. Class balancing      (this module)
  B. Improve the result   (this module)
  C. Feature importance   (feature_importance.py)

PROTOCOL: identical to the reproduction in ensemble.run_repeated_cv --
stratified 10-fold CV x N_REPEATS with fold seeds 0..N_REPEATS-1, and every
scaler / resampler inside the pipeline so it is fit on the training fold only.
Every Review-2 number is therefore directly comparable with Review 1, and the
"partial" balancing configuration below reproduces the Review-1 ensemble
exactly (checked at run time).

OUTPUT: results/review2/{tables,figures}/ -- never results/tables/, so the
Review-1 reproduction numbers stay byte-identical and remain the baseline.

PRIMARY METRIC: macro-F1, fixed in config.REVIEW2_PRIMARY_METRIC before any of
these experiments was run. Accuracy is reported alongside it throughout.

Experiment A -- class balancing
-------------------------------
The same four learners and the paper's weighted soft vote, under four
balancing schemes, so the balancing is the only thing that changes:

  none          no balancing anywhere
  partial       Review 1: balanced class weights on SVM and RF only
  class_weight  balanced weights on all four (MLP and GradBoost have no
                class_weight parameter, so they get the equivalent per-sample
                weights through BalancedSampleWeight)
  smote         SMOTE oversampling of the training fold, no class weights

Experiment B -- improving the result
------------------------------------
  B1  Weight sharpening. Review 1 diagnosed that Eqs. (5) and (9) produce
      near-uniform weights, so the ensemble averages instead of exceeding its
      best member, and predicted that sharper weights should close the gap to
      Random Forest. A temperature T on the two normalisations tests that
      directly. This is a DIAGNOSTIC: the weights are computed from the same
      out-of-fold predictions they are scored on, so a T picked from this
      sweep is never reported as the improved result.
  B2  Stacking: replace the hand-built weights with a learned combiner (a
      logistic regression trained on the members' out-of-fold probabilities
      by sklearn's StackingClassifier, which runs its own inner 5-fold CV, so
      the combiner never sees the outer test fold). Same four members.
  B3  Stronger members: ExtraTrees and HistGradientBoosting, alone and
      stacked with Random Forest and SVM.
  B4  Learned temperature -- the fix the Review-1 diagnosis points to. The
      paper's ensemble, but the temperature (and whether it sharpens one term
      or both) is CHOSEN INSIDE EACH TRAINING FOLD by an inner 5-fold CV on
      the primary metric. The choice never sees the outer test fold, so --
      unlike the B1 sweep -- this is a legitimate result.
  B5  Stacking with a class-balanced combiner.

  B4 and B5 were added after a 1-repeat smoke test, which showed (i) that
  sharpening the weights helps, exactly as Review 1 predicted, and (ii) that
  the plain stacking combiner learns the majority-class bias back and discards
  the balancing. Both are principled fixes rather than tuning: B4's
  temperature is selected by nested CV, never from the B1 sweep, and B5 only
  makes the combiner consistent with its class-weighted members. The reported
  numbers come from the full 10-repeat run, not the smoke test.
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from imblearn.over_sampling import SMOTE  # noqa: E402
from imblearn.pipeline import Pipeline as ImbPipeline  # noqa: E402
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # noqa: E402
from sklearn.ensemble import (  # noqa: E402
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import MinMaxScaler  # noqa: E402
from sklearn.utils.class_weight import compute_sample_weight  # noqa: E402

from base_learners import get_base_learners  # noqa: E402
from config import (  # noqa: E402
    N_SPLITS,
    RANDOM_STATE,
    REVIEW2_PRIMARY_METRIC,
    SMOTE_K_NEIGHBORS,
    WEIGHT_TEMPERATURES,
)
from ensemble import _error_vector, weighted_soft_vote  # noqa: E402

# Colours shared with the slide deck, so figures and slides read as one piece.
BERRY, ROSE, GOLD, INK, MUTED, SAGE = "#6D2E46", "#A26769", "#C8A45C", "#2B2B2B", "#9A9A9A", "#5B7F6B"

BALANCING_METHODS = ["none", "partial", "class_weight", "smote"]
BALANCING_LABELS = {
    "none": "No balancing",
    "partial": "Review 1 (SVM + RF only)",
    "class_weight": "Class weights, all four",
    "smote": "SMOTE oversampling",
}
PAPER_ENSEMBLE = "Ensemble (paper weights)"
MIXED_PAPER_ENSEMBLE = "Paper weights: SVM + RF + ExtraTrees + HistGB"
STACK_PAPER4 = "Stacked: paper's 4 learners (class-weighted)"
STACK_STRONG = "Stacked: SVM + RF + ExtraTrees + HistGB"
STACK_BALANCED = "Stacked: paper's 4 learners, balanced combiner"
LEARNED_T = "Learned-temperature ensemble"
LEARNED_T_CONFIGS = ["partial", "smote"]


# --------------------------------------------------------------------------
# Building blocks
# --------------------------------------------------------------------------
class BalancedSampleWeight(ClassifierMixin, BaseEstimator):
    """Fit `estimator` with 'balanced' per-sample weights computed from the
    labels it is given -- i.e. from the training fold only.

    GradientBoostingClassifier and MLPClassifier have no class_weight
    parameter, but both accept sample_weight in fit(). Giving each sample the
    weight n_samples / (n_classes * count(its class)) is exactly what
    class_weight='balanced' does inside SVM and Random Forest.
    """

    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
        self.classes_ = self.estimator_.classes_
        return self

    def predict(self, X):
        return self.estimator_.predict(X)

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)


def _pipe(estimator):
    return Pipeline([("scaler", MinMaxScaler()), ("clf", estimator)])


def make_paper_learners(balancing, random_state=RANDOM_STATE):
    """The reproduction's four learners under one balancing scheme.

    Everything is derived from base_learners.get_base_learners(), so the
    hyperparameters are identical to Review 1 and only the balancing changes.
    """
    learners = get_base_learners(random_state)
    if balancing == "partial":
        return learners

    if balancing in ("none", "smote"):
        learners["SVM"].set_params(clf__estimator__class_weight=None)
        learners["RF"].set_params(clf__class_weight=None)
    if balancing == "none":
        return learners

    if balancing == "class_weight":
        for name in ("MLP", "GradBoost"):
            learners[name].set_params(clf=BalancedSampleWeight(learners[name].named_steps["clf"]))
        return learners

    if balancing == "smote":
        # Oversample AFTER scaling (SMOTE interpolates between neighbours, so
        # it needs comparable feature scales) and only inside fit(), so the
        # test fold is never resampled.
        return {
            name: ImbPipeline(
                [
                    ("scaler", MinMaxScaler()),
                    ("smote", SMOTE(k_neighbors=SMOTE_K_NEIGHBORS, random_state=random_state)),
                    ("clf", pipe.named_steps["clf"]),
                ]
            )
            for name, pipe in learners.items()
        }

    raise ValueError(f"unknown balancing method: {balancing!r}")


def make_strong_learners(random_state=RANDOM_STATE):
    """Two members not in the paper, both class-weighted."""
    return {
        "ExtraTrees": _pipe(
            ExtraTreesClassifier(
                n_estimators=500, class_weight="balanced_subsample", random_state=random_state
            )
        ),
        "HistGB": _pipe(HistGradientBoostingClassifier(class_weight="balanced", random_state=random_state)),
    }


def make_stack(members, balanced_combiner=False, random_state=RANDOM_STATE):
    """A learned combiner in place of the paper's hand-built weights."""
    return StackingClassifier(
        estimators=list(members.items()),
        final_estimator=LogisticRegression(
            max_iter=5000, class_weight="balanced" if balanced_combiner else None
        ),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state),
        stack_method="predict_proba",
        n_jobs=1,
    )


def strong_member_set(random_state=RANDOM_STATE):
    """SVM and RF (class-weighted) plus the two stronger learners."""
    cw = make_paper_learners("class_weight", random_state)
    return {"SVM": cw["SVM"], "RF": cw["RF"], **make_strong_learners(random_state)}


def _metric(y_true, y_pred, name):
    if name == "macro_f1":
        return f1_score(y_true, y_pred, average="macro", zero_division=0)
    if name == "accuracy":
        return accuracy_score(y_true, y_pred)
    raise ValueError(name)


class PaperWeightedEnsemble(ClassifierMixin, BaseEstimator):
    """Ma et al.'s weighted soft vote as an ordinary fit / predict estimator.

    In the cross-validation experiments the paper's weights are computed from
    each repeat's out-of-fold predictions, exactly as in Review 1. A standalone
    estimator cannot do that, so here the weights come from an inner
    cross-validation on the training data it is fitted on -- the same
    procedure, applied one level down.

    With the default temperatures=(1,) this is exactly the paper's scheme.
    Given several temperatures it becomes the Review-2 fix (B4): every
    (temperature, sharpen) pair is scored on the inner out-of-fold predictions
    and the best on `selection_metric` is kept. Ties go to the smaller
    temperature -- i.e. towards the paper's own setting.
    """

    def __init__(self, members, temperatures=(1,), sharpen_modes=("both",),
                 selection_metric=REVIEW2_PRIMARY_METRIC, inner_splits=5, random_state=RANDOM_STATE):
        self.members = members
        self.temperatures = temperatures
        self.sharpen_modes = sharpen_modes
        self.selection_metric = selection_metric
        self.inner_splits = inner_splits
        self.random_state = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        skf = StratifiedKFold(n_splits=self.inner_splits, shuffle=True, random_state=self.random_state)
        probas = {
            name: cross_val_predict(clone(model), X, y, cv=skf, method="predict_proba")
            for name, model in self.members.items()
        }
        errors = {n: _error_vector(y, self.classes_[p.argmax(axis=1)]) for n, p in probas.items()}

        best = None
        for T in sorted(self.temperatures):
            for mode in self.sharpen_modes:
                w = tempered_weights(errors, T, mode)
                pred = weighted_soft_vote(probas, w, list(self.classes_))
                value = _metric(y, pred, self.selection_metric)
                if best is None or value > best[0] + 1e-12:
                    best = (value, T, mode, w)
        self.inner_score_, self.temperature_, self.sharpen_, self.weights_ = best
        self.fitted_ = {name: clone(model).fit(X, y) for name, model in self.members.items()}
        return self

    def predict_proba(self, X):
        return sum(self.weights_[n] * m.predict_proba(X) for n, m in self.fitted_.items())

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


def make_learned_temperature(balancing, random_state=RANDOM_STATE):
    """B4: the paper's ensemble with the temperature chosen by inner CV."""
    return PaperWeightedEnsemble(
        make_paper_learners(balancing, random_state),
        temperatures=tuple(WEIGHT_TEMPERATURES),
        sharpen_modes=("both", "accuracy"),
        random_state=random_state,
    )


def estimator_for(model_name):
    """Map a row of model_comparison.csv back to an unfitted estimator."""
    if " | " in model_name:
        member, label = model_name.split(" | ")
        balancing = {v: k for k, v in BALANCING_LABELS.items()}[label]
        if member == LEARNED_T:
            return make_learned_temperature(balancing)
        learners = make_paper_learners(balancing)
        return PaperWeightedEnsemble(learners) if member == PAPER_ENSEMBLE else learners[member]
    if model_name in make_strong_learners():
        return make_strong_learners()[model_name]
    if model_name == MIXED_PAPER_ENSEMBLE:
        return PaperWeightedEnsemble(strong_member_set())
    if model_name == STACK_PAPER4:
        return make_stack(make_paper_learners("class_weight"))
    if model_name == STACK_STRONG:
        return make_stack(strong_member_set())
    if model_name == STACK_BALANCED:
        return make_stack(make_paper_learners("class_weight"), balanced_combiner=True)
    raise KeyError(f"no estimator registered for {model_name!r}")


def collect_oof_probas(X, y, learners, n_repeats, label="", n_splits=N_SPLITS, base_seed=0):
    """Out-of-fold predict_proba for every learner, per repeat, on exactly the
    folds ensemble.run_repeated_cv uses (same seeds), so results line up with
    Review 1 sample for sample."""
    per_repeat = []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=base_seed + r)
        per_repeat.append(
            {
                name: cross_val_predict(model, X, y, cv=skf, n_jobs=-1, method="predict_proba")
                for name, model in learners.items()
            }
        )
        print(f"    [{label}] repeat {r + 1}/{n_repeats}", flush=True)
    return per_repeat


def labels_from(per_repeat, name, classes):
    return np.concatenate([np.asarray(classes)[p[name].argmax(axis=1)] for p in per_repeat])


def tempered_weights(error_vectors, temperature=1, sharpen="both"):
    """Eqs. (4)-(9) of Ma et al. with a temperature T on the normalisations:

        w_acc_i ∝ (1 / eps_i)^T      -- Eq. (5) at T = 1
        w_div_i ∝ D_i^T              -- Eq. (9) at T = 1
        w_i      = (w_acc_i + w_div_i) / 2   -- Eq. (4)

    T = 1 reproduces the paper exactly (x ** 1 is exact in floating point).
    `sharpen="accuracy"` applies T to the accuracy term only, which separates
    the effect of the two halves of the scheme.
    """
    names = list(error_vectors)
    t_acc = temperature
    t_div = temperature if sharpen == "both" else 1

    cv_err = {n: float(error_vectors[n].mean()) for n in names}
    inv = {n: (1.0 / cv_err[n] if cv_err[n] > 0 else 1e6) ** t_acc for n in names}
    total_inv = sum(inv.values())
    w_acc = {n: inv[n] / total_inv for n in names}

    D = {
        n: sum(np.linalg.norm(error_vectors[n] - error_vectors[o]) for o in names if o != n)
        for n in names
    }
    Dt = {n: D[n] ** t_div for n in names}
    total_D = sum(Dt.values())
    w_div = {n: Dt[n] / total_D for n in names}

    w = {n: 0.5 * (w_acc[n] + w_div[n]) for n in names}
    if not all(np.isfinite(v) for v in w.values()):
        raise FloatingPointError(f"non-finite weights at T={temperature}")
    return w


def paper_weighted_ensemble(per_repeat, y, classes, temperature=1, sharpen="both"):
    """The paper's weighted soft vote, recomputed per repeat as in Review 1."""
    preds, weights = [], []
    for probas in per_repeat:
        fold_preds = {n: np.asarray(classes)[p.argmax(axis=1)] for n, p in probas.items()}
        errs = {n: _error_vector(y, fold_preds[n]) for n in probas}
        w = tempered_weights(errs, temperature, sharpen)
        preds.append(weighted_soft_vote(probas, w, classes))
        weights.append(w)
    return np.concatenate(preds), weights


def score(y_true, y_pred, classes):
    """Accuracy, balanced accuracy, macro / weighted F1 and per-class recall (%).

    Per-class recall is the metric Ma et al. call "precision" -- see the
    terminology note in evaluate.py.
    """
    recall = recall_score(y_true, y_pred, labels=classes, average=None, zero_division=0)
    row = {
        "accuracy": 100 * accuracy_score(y_true, y_pred),
        "balanced_accuracy": 100 * balanced_accuracy_score(y_true, y_pred),
        "macro_f1": 100 * f1_score(y_true, y_pred, labels=classes, average="macro", zero_division=0),
        "weighted_f1": 100 * f1_score(y_true, y_pred, labels=classes, average="weighted", zero_division=0),
    }
    for c, r in zip(classes, recall):
        row[f"recall_{c}"] = 100 * r
    return row


# --------------------------------------------------------------------------
# Experiment A -- class balancing
# --------------------------------------------------------------------------
def experiment_a_class_balancing(X, y, classes, n_repeats, tables_dir, figures_dir):
    y_pooled = np.concatenate([y] * n_repeats)
    rows, cache = [], {}

    preds = {}
    for b in BALANCING_METHODS:
        print(f"  A. balancing = {b}", flush=True)
        learners = make_paper_learners(b)
        per_repeat = collect_oof_probas(X, y, learners, n_repeats, label=b)
        cache[b] = per_repeat
        for name in learners:
            pred = labels_from(per_repeat, name, classes)
            preds[f"{name} | {BALANCING_LABELS[b]}"] = pred
            rows.append({"balancing": b, "model": name, **score(y_pooled, pred, classes)})
        ens, _ = paper_weighted_ensemble(per_repeat, y, classes)
        preds[f"{PAPER_ENSEMBLE} | {BALANCING_LABELS[b]}"] = ens
        rows.append({"balancing": b, "model": PAPER_ENSEMBLE, **score(y_pooled, ens, classes)})

    df = pd.DataFrame(rows)
    df.insert(1, "balancing_label", df["balancing"].map(BALANCING_LABELS))
    df.round(4).to_csv(tables_dir / "A_class_balancing.csv", index=False)

    ens_rows = df[df["model"] == PAPER_ENSEMBLE].set_index("balancing").loc[BALANCING_METHODS]
    _plot_balancing(ens_rows, classes, figures_dir)
    return df, cache, preds


def _plot_balancing(ens_rows, classes, figures_dir):
    labels = [BALANCING_LABELS[b] for b in ens_rows.index]
    colours = [MUTED, ROSE, BERRY, GOLD]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    x = np.arange(len(labels))
    ax.bar(x - 0.2, ens_rows["accuracy"], 0.4, label="Accuracy", color=MUTED)
    ax.bar(x + 0.2, ens_rows["macro_f1"], 0.4, label="Macro-F1 (primary)", color=BERRY)
    for i, (a, f) in enumerate(zip(ens_rows["accuracy"], ens_rows["macro_f1"])):
        ax.text(i - 0.2, a + 0.8, f"{a:.1f}", ha="center", fontsize=9)
        ax.text(i + 0.2, f + 0.8, f"{f:.1f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=12, ha="right")
    ax.set_ylabel("%")
    ax.set_ylim(0, 85)
    ax.set_title("Paper-weighted ensemble under each balancing scheme")
    ax.legend(frameon=False)

    ax = axes[1]
    cx = np.arange(len(classes))
    width = 0.2
    for i, (b, colour) in enumerate(zip(ens_rows.index, colours)):
        vals = [ens_rows.loc[b, f"recall_{c}"] for c in classes]
        ax.bar(cx + (i - 1.5) * width, vals, width, label=BALANCING_LABELS[b], color=colour)
    ax.set_xticks(cx)
    ax.set_xticklabels([str(c) for c in classes])
    ax.set_xlabel("Quality class")
    ax.set_ylabel("Recall (%)  [the paper's 'precision']")
    ax.set_ylim(0, 100)
    ax.set_title("Per-class recall: where balancing helps")
    ax.legend(frameon=False, fontsize=8)

    for a in axes:
        a.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figures_dir / "A_class_balancing.png", dpi=170)
    plt.close(fig)


# --------------------------------------------------------------------------
# Experiment B -- improving the result
# --------------------------------------------------------------------------
def experiment_b1_temperature(cache, y, classes, configs, tables_dir, figures_dir):
    """Diagnostic sweep -- see module docstring for why it is never used to
    pick the final model."""
    n_repeats = len(next(iter(cache.values())))
    y_pooled = np.concatenate([y] * n_repeats)
    rows = []
    for cfg in configs:
        rf_acc = score(y_pooled, labels_from(cache[cfg], "RF", classes), classes)["accuracy"]
        for sharpen in ("both", "accuracy"):
            for T in WEIGHT_TEMPERATURES:
                ens, ws = paper_weighted_ensemble(cache[cfg], y, classes, temperature=T, sharpen=sharpen)
                mean_w = {n: float(np.mean([w[n] for w in ws])) for n in ws[0]}
                top = max(mean_w, key=mean_w.get)
                rows.append(
                    {
                        "balancing": cfg,
                        "sharpen": sharpen,
                        "temperature": T,
                        **score(y_pooled, ens, classes),
                        "rf_accuracy": rf_acc,
                        "largest_weight_learner": top,
                        "largest_weight": mean_w[top],
                        **{f"w_{n}": v for n, v in mean_w.items()},
                    }
                )
    df = pd.DataFrame(rows)
    df.round(4).to_csv(tables_dir / "B1_weight_temperature.csv", index=False)
    _plot_temperature(df, figures_dir)
    return df


def _plot_temperature(df, figures_dir):
    cfgs = list(dict.fromkeys(df["balancing"]))
    fig, axes = plt.subplots(1, len(cfgs), figsize=(6.5 * len(cfgs), 4.6), squeeze=False)
    for ax, cfg in zip(axes[0], cfgs):
        sub = df[df["balancing"] == cfg]
        for sharpen, colour, label in (
            ("accuracy", BERRY, "Sharpen accuracy term only"),
            ("both", GOLD, "Sharpen both terms"),
        ):
            s = sub[sub["sharpen"] == sharpen]
            ax.plot(s["temperature"], s["accuracy"], marker="o", color=colour, label=label)
        ax.axhline(sub["rf_accuracy"].iloc[0], color=INK, ls="--", lw=1, label="Random Forest alone")
        ax.set_xscale("log")
        ax.set_xticks(sorted(sub["temperature"].unique()))
        ax.set_xticklabels([str(t) for t in sorted(sub["temperature"].unique())])
        ax.set_xlabel("Temperature T  (T = 1 is the paper)")
        ax.set_ylabel("Ensemble accuracy (%)")
        ax.set_title(f"Weight sharpening -- {BALANCING_LABELS[cfg]}")
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(figures_dir / "B1_weight_temperature.png", dpi=170)
    plt.close(fig)


def experiment_b2_b3_improvements(X, y, classes, n_repeats, cache, tables_dir):
    """Stacking (same four members) and stronger members."""
    y_pooled = np.concatenate([y] * n_repeats)
    rows = []
    cw = make_paper_learners("class_weight")
    strong = make_strong_learners()

    preds = {}
    print("  B3. stronger single learners", flush=True)
    strong_probas = collect_oof_probas(X, y, strong, n_repeats, label="strong singles")
    for name in strong:
        preds[name] = labels_from(strong_probas, name, classes)
        rows.append({"model": name, "kind": "single learner", **score(y_pooled, preds[name], classes)})

    # The paper's weighting scheme, applied to the stronger member set
    # (probabilities reused -- no refitting).
    mixed = [
        {"SVM": c["SVM"], "RF": c["RF"], "ExtraTrees": s["ExtraTrees"], "HistGB": s["HistGB"]}
        for c, s in zip(cache["class_weight"], strong_probas)
    ]
    ens, _ = paper_weighted_ensemble(mixed, y, classes)
    preds[MIXED_PAPER_ENSEMBLE] = ens
    rows.append({"model": MIXED_PAPER_ENSEMBLE, "kind": "ensemble", **score(y_pooled, ens, classes)})

    stacks = {
        STACK_PAPER4: make_stack(cw),
        STACK_BALANCED: make_stack(cw, balanced_combiner=True),
        STACK_STRONG: make_stack(strong_member_set()),
    }
    for name, stack in stacks.items():
        print(f"  B2. {name}", flush=True)
        probas = collect_oof_probas(X, y, {name: stack}, n_repeats, label="stack")
        preds[name] = labels_from(probas, name, classes)
        rows.append({"model": name, "kind": "ensemble", **score(y_pooled, preds[name], classes)})

    print("  B4. learned-temperature ensemble (temperature chosen by inner CV)", flush=True)
    chosen_rows = []
    for balancing in LEARNED_T_CONFIGS:
        name = f"{LEARNED_T} | {BALANCING_LABELS[balancing]}"
        pred, chosen = _oof_learned_temperature(X, y, balancing, n_repeats)
        preds[name] = pred
        rows.append({"model": name, "kind": "ensemble", **score(y_pooled, pred, classes)})
        chosen_rows += [{"balancing": balancing, **c} for c in chosen]

    chosen_df = pd.DataFrame(chosen_rows)
    chosen_df.to_csv(tables_dir / "B4_chosen_temperatures.csv", index=False)
    plot_chosen_temperatures(chosen_df, tables_dir.parent / "figures")

    df = pd.DataFrame(rows)
    df.round(4).to_csv(tables_dir / "B2_B3_improvements.csv", index=False)
    return df, preds


def plot_chosen_temperatures(chosen_df, figures_dir):
    """How often each outer fold's inner CV picked each temperature."""
    temps = sorted(chosen_df["temperature"].unique())
    fig, ax = plt.subplots(figsize=(6.2, 4.3))
    x = np.arange(len(temps))
    for i, (b, colour, label) in enumerate((("partial", BERRY, "Review-1 members"), ("smote", GOLD, "SMOTE members"))):
        sub = chosen_df[chosen_df["balancing"] == b]
        if sub.empty:
            continue
        counts = [int((sub["temperature"] == t).sum()) for t in temps]
        bars = ax.bar(x + (i - 0.5) * 0.38, counts, 0.38, color=colour, label=label)
        ax.bar_label(bars, fontsize=8, padding=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f"T = {t:g}" + ("\n(paper)" if t == 1 else "") for t in temps], fontsize=9)
    ax.set_ylabel("Number of folds")
    ax.set_title("Temperature chosen by each fold's inner CV", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(figures_dir / "B4_chosen_temperatures.png", dpi=170)
    plt.close(fig)


def _oof_learned_temperature(X, y, balancing, n_repeats, n_splits=N_SPLITS):
    """Pooled out-of-fold predictions for B4 on the Review-1 folds, plus the
    temperature each outer fold's inner CV chose."""
    preds, chosen = [], []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=r)
        cv = cross_validate(make_learned_temperature(balancing), X, y, cv=skf,
                            return_estimator=True, return_indices=True, n_jobs=-1)
        oof = np.empty_like(y)
        for fold, (est, test) in enumerate(zip(cv["estimator"], cv["indices"]["test"])):
            oof[test] = est.predict(X[test])
            chosen.append({"repeat": r, "fold": fold, "temperature": est.temperature_,
                           "sharpen": est.sharpen_, "inner_score": est.inner_score_,
                           **{f"w_{n}": v for n, v in est.weights_.items()}})
        preds.append(oof)
        print(f"    [learned T, {balancing}] repeat {r + 1}/{n_repeats}", flush=True)
    return np.concatenate(preds), chosen


# --------------------------------------------------------------------------
# Putting A and B together
# --------------------------------------------------------------------------
def build_model_comparison(a_df, b_df, tables_dir):
    """Every candidate the project evaluated, ranked on the primary metric.

    Selection is made on the same cross-validation that scores the candidates,
    so the winner's score carries a small optimistic bias; with this few
    candidates it is small, and it is disclosed rather than hidden. The
    temperature sweep (B1) is excluded on purpose -- see the module docstring.
    """
    a_part = a_df.copy()
    a_part["model"] = a_part.apply(
        lambda r: f"{r['model']} | {BALANCING_LABELS[r['balancing']]}", axis=1
    )
    a_part["kind"] = np.where(
        a_part["model"].str.startswith(PAPER_ENSEMBLE), "ensemble", "single learner"
    )
    a_part["experiment"] = "A. class balancing"
    b_part = b_df.copy()
    b_part["experiment"] = "B. improvements"

    cols = ["experiment", "model", "kind"] + [c for c in a_df.columns if c not in ("balancing", "balancing_label", "model")]
    comp = pd.concat([a_part[cols], b_part[cols]], ignore_index=True)
    comp = comp.sort_values(REVIEW2_PRIMARY_METRIC, ascending=False).reset_index(drop=True)
    comp.insert(0, "rank_by_" + REVIEW2_PRIMARY_METRIC, comp.index + 1)
    comp.round(4).to_csv(tables_dir / "model_comparison.csv", index=False)
    return comp


def headline_table(a_df, comp, tables_dir):
    """Review 1 baseline vs the best Review-2 model, side by side."""
    base = a_df[(a_df["balancing"] == "partial") & (a_df["model"] == PAPER_ENSEMBLE)].iloc[0]
    rf = a_df[(a_df["balancing"] == "partial") & (a_df["model"] == "RF")].iloc[0]
    best = comp.iloc[0]
    metrics = [c for c in comp.columns if c in base.index and c not in ("model", "balancing", "balancing_label")]
    out = pd.DataFrame(
        {
            "metric": metrics,
            "review1_ensemble": [base[m] for m in metrics],
            "review1_random_forest": [rf[m] for m in metrics],
            "review2_best": [best[m] for m in metrics],
        }
    )
    out["change_vs_review1_ensemble"] = out["review2_best"] - out["review1_ensemble"]
    out.round(4).to_csv(tables_dir / "headline_review1_vs_review2.csv", index=False)
    with open(tables_dir / "headline_best_model.txt", "w") as f:
        f.write(str(best["model"]) + "\n")
    return out, best["model"]


def plot_headline(comp, figures_dir, top_n=8):
    ens = comp[comp["kind"] == "ensemble"].head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 0.55 * len(ens) + 1.6))
    y_pos = np.arange(len(ens))
    ax.barh(y_pos - 0.2, ens["macro_f1"], 0.4, color=BERRY, label="Macro-F1 (primary)")
    ax.barh(y_pos + 0.2, ens["accuracy"], 0.4, color=MUTED, label="Accuracy")
    for i, (f, a) in enumerate(zip(ens["macro_f1"], ens["accuracy"])):
        ax.text(f + 0.4, i - 0.2, f"{f:.2f}", va="center", fontsize=8)
        ax.text(a + 0.4, i + 0.2, f"{a:.2f}", va="center", fontsize=8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([m.replace(" | ", "\n") for m in ens["model"]], fontsize=8)
    ax.set_xlabel("%")
    ax.set_xlim(0, 85)
    ax.set_title("Ensembles ranked by macro-F1")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(figures_dir / "B_model_comparison.png", dpi=170)
    plt.close(fig)


def plot_tradeoff(comp, figures_dir):
    """Every candidate as a point: accuracy (x) against macro-F1 (y).

    The upper-right boundary (the Pareto frontier) is the set of models no
    other model beats on BOTH metrics at once -- the honest answer to "which
    model is best" when the two metrics disagree.
    """
    d = comp.copy()
    pts = d[["accuracy", "macro_f1"]].to_numpy()
    on_front = np.array([
        not np.any((pts[:, 0] >= a) & (pts[:, 1] >= f) & ((pts[:, 0] > a) | (pts[:, 1] > f)))
        for a, f in pts
    ])
    d["pareto"] = on_front

    fig, ax = plt.subplots(figsize=(10.5, 6.2))
    singles = d[d["kind"] == "single learner"]
    ensembles = d[d["kind"] == "ensemble"]
    ax.scatter(singles["accuracy"], singles["macro_f1"], s=36, color=MUTED, alpha=0.75, label="Single learner")
    ax.scatter(ensembles["accuracy"], ensembles["macro_f1"], s=52, color=ROSE, alpha=0.9, label="Ensemble")

    front = d[d["pareto"]].sort_values("accuracy")
    ax.plot(front["accuracy"], front["macro_f1"], color=BERRY, lw=1.6, ls="--", label="Best trade-off frontier")
    ax.scatter(front["accuracy"], front["macro_f1"], s=80, facecolor="none", edgecolor=BERRY, lw=1.6)

    def mark(name, label, colour, dx, dy):
        row = d[d["model"] == name]
        if row.empty:
            return
        a, f = row["accuracy"].iloc[0], row["macro_f1"].iloc[0]
        ax.scatter([a], [f], s=95, color=colour, zorder=6, edgecolor=INK, lw=0.6)
        ax.annotate(label, (a, f), textcoords="offset points", xytext=(dx, dy), fontsize=8.5,
                    color=INK, arrowprops={"arrowstyle": "-", "color": MUTED, "lw": 0.7})

    mark(f"{PAPER_ENSEMBLE} | {BALANCING_LABELS['partial']}", "Review 1 ensemble", GOLD, -95, -22)
    mark(f"RF | {BALANCING_LABELS['partial']}", "Review 1 Random Forest", GOLD, 10, -26)
    best = d.sort_values(REVIEW2_PRIMARY_METRIC, ascending=False).iloc[0]
    mark(best["model"], "Best on macro-F1", BERRY, -120, 10)
    top_acc = d.sort_values("accuracy", ascending=False).iloc[0]
    mark(top_acc["model"], "Best on accuracy", SAGE, -10, 22)

    # Zoom to where the ensembles sit; say how many weaker points fall off the
    # left edge rather than dropping them silently.
    x_lo = float(np.floor(ensembles["accuracy"].min() - 2))
    y_lo = float(np.floor(d.loc[d["accuracy"] >= x_lo, "macro_f1"].min() - 1))
    hidden = int((d["accuracy"] < x_lo).sum())
    ax.set_xlim(x_lo, float(np.ceil(d["accuracy"].max() + 1)))
    ax.set_ylim(y_lo, float(np.ceil(d["macro_f1"].max() + 1)))
    if hidden:
        ax.text(0.01, 0.01, f"{hidden} weaker single learners (accuracy < {x_lo:.0f}%) lie off the left edge",
                transform=ax.transAxes, fontsize=8, color=MUTED, ha="left", va="bottom")

    ax.set_xlabel("Accuracy (%)  -- rewards getting the common classes right")
    ax.set_ylabel("Macro-F1 (%)  -- weights all six classes equally")
    ax.set_title("No model wins both: the accuracy vs. rare-class trade-off")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8.5, loc="upper left")
    fig.tight_layout()
    fig.savefig(figures_dir / "B_tradeoff.png", dpi=170)
    plt.close(fig)
    return d


def per_repeat_scores(all_preds, y, classes, n_repeats):
    """Split every candidate's pooled predictions back into its 10 repeats.

    Pooled predictions are the concatenation of the repeats in order (seed 0,
    1, ...), so slice r is exactly repeat r -- and every candidate was scored
    on the same folds in the same repeat, which makes the comparison paired.
    """
    n = len(y)
    rows = []
    for name, pred in all_preds.items():
        for r in range(n_repeats):
            s = score(y, pred[r * n:(r + 1) * n], classes)
            rows.append({"model": name, "repeat": r, "accuracy": s["accuracy"],
                         "balanced_accuracy": s["balanced_accuracy"], "macro_f1": s["macro_f1"]})
    return pd.DataFrame(rows)


def paired_comparison(per_rep, pairs, tables_dir):
    """Is a difference consistent across the repeats, or noise?

    For each (model, comparator) pair: the mean and spread of the per-repeat
    difference, how many repeats the model wins, and a Wilcoxon signed-rank
    test. CAVEAT: the repeats reuse the same 1,599 wines with different splits,
    so they are not independent samples and the p-value is optimistic. The
    win count is the plainer evidence; the p-value is indicative only.
    """
    from scipy.stats import wilcoxon

    rows = []
    for model, other in pairs:
        a = per_rep[per_rep["model"] == model].sort_values("repeat")
        b = per_rep[per_rep["model"] == other].sort_values("repeat")
        if a.empty or b.empty or model == other:
            continue
        for metric in ("macro_f1", "accuracy"):
            d = a[metric].to_numpy() - b[metric].to_numpy()
            p = float(wilcoxon(d).pvalue) if len(d) > 1 and np.any(d != 0) else np.nan
            rows.append({"model": model, "compared_with": other, "metric": metric,
                         "mean_difference": d.mean(), "sd_difference": d.std(ddof=1) if len(d) > 1 else np.nan,
                         "repeats_won": int((d > 0).sum()), "repeats_tied": int((d == 0).sum()),
                         "repeats": len(d), "wilcoxon_p": p})
    out = pd.DataFrame(rows)
    out.round(6).to_csv(tables_dir / "paired_comparison.csv", index=False)
    return out


def save_confusion(y_true, y_pred, classes, name, tables_dir):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes]).to_csv(
        tables_dir / f"confusion_matrix_{name}.csv"
    )
    return cm

In [ ]:
%%writefile src/feature_importance.py
"""
feature_importance.py

Review 2, Experiment C -- which of the 11 lab measurements drive the model?

No single importance measure is trustworthy on its own, so three are used and
checked against each other:

  1. PERMUTATION IMPORTANCE (primary). Shuffle one feature inside the held-out
     test fold and measure how much macro-F1 falls. It is model-agnostic and is
     computed on data the model never trained on, so it measures what the
     model actually relies on. Averaged over the 10 test folds, with
     PERMUTATION_REPEATS shuffles per feature per fold.
  2. RANDOM FOREST IMPURITY IMPORTANCE (MDI). Comes free with the forest but is
     known to favour continuous, many-valued features -- shown for comparison,
     never as the headline.
  3. DIRECTION. Spearman rank correlation of each feature with quality on the
     full dataset. Importance says how much a feature matters; this says which
     way it pushes quality. It is descriptive and never feeds a prediction.

Then a practical test: retrain with only the top-k features, chosen INSIDE each
training fold. Choosing them on the full dataset first would leak test-fold
information into the choice. It answers "how many lab tests are needed?", plus
a direct check of the four attributes Ma et al. highlight (Figs. 4-5).
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.patches import Patch  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from joblib import Parallel, delayed  # noqa: E402
from scipy.stats import spearmanr  # noqa: E402
from sklearn.base import clone  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.feature_selection import SelectFromModel  # noqa: E402
from sklearn.inspection import permutation_importance  # noqa: E402
from sklearn.model_selection import StratifiedKFold, cross_validate  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import MinMaxScaler  # noqa: E402

from base_learners import get_base_learners  # noqa: E402
from config import (  # noqa: E402
    FEATURE_SUBSET_SIZES,
    N_SPLITS,
    PAPER_KEY_ATTRIBUTES,
    PERMUTATION_REPEATS,
    RANDOM_STATE,
)
from review2 import BERRY, GOLD, INK, MUTED, ROSE, SAGE, estimator_for, score  # noqa: E402


# --------------------------------------------------------------------------
# 1 + 2. Permutation importance (and MDI where the model has it)
# --------------------------------------------------------------------------
def _one_fold(model, X, y, train, test, fold, seed):
    fitted = clone(model).fit(X[train], y[train])
    result = permutation_importance(
        fitted,
        X[test],
        y[test],
        scoring=["f1_macro", "accuracy"],
        n_repeats=PERMUTATION_REPEATS,
        random_state=seed + fold,
        n_jobs=1,
    )
    mdi = None
    clf = getattr(fitted, "named_steps", {}).get("clf")
    if clf is not None and hasattr(clf, "feature_importances_"):
        mdi = clf.feature_importances_
    return result["f1_macro"].importances_mean, result["accuracy"].importances_mean, mdi


def permutation_importance_cv(model, X, y, feature_names, seed=0, n_splits=N_SPLITS):
    """Permutation importance on each held-out fold of one stratified 10-fold
    split, in percentage points of macro-F1 / accuracy lost."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = Parallel(n_jobs=-1)(
        delayed(_one_fold)(model, X, y, tr, te, k, seed) for k, (tr, te) in enumerate(skf.split(X, y))
    )
    f1 = 100 * np.array([f[0] for f in folds])
    acc = 100 * np.array([f[1] for f in folds])
    out = pd.DataFrame(
        {
            "feature": feature_names,
            "f1_drop_mean": f1.mean(axis=0),
            "f1_drop_std": f1.std(axis=0),
            "accuracy_drop_mean": acc.mean(axis=0),
            "accuracy_drop_std": acc.std(axis=0),
        }
    )
    if all(f[2] is not None for f in folds):
        out["mdi_mean"] = np.array([f[2] for f in folds]).mean(axis=0)
    return out


# --------------------------------------------------------------------------
# 3. Direction of effect
# --------------------------------------------------------------------------
def direction_of_effect(X, y, feature_names):
    rows = []
    for j, name in enumerate(feature_names):
        rho, p = spearmanr(X[:, j], y)
        rows.append({"feature": name, "spearman_rho": rho, "p_value": p})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Practical test: how many lab tests are needed?
# --------------------------------------------------------------------------
def _rf(n_estimators):
    return RandomForestClassifier(
        n_estimators=n_estimators, class_weight="balanced_subsample", random_state=RANDOM_STATE
    )


def _oof_with_selection(model, X, y, n_repeats, n_splits=N_SPLITS):
    """Pooled out-of-fold predictions on the Review-1 folds, plus which
    features the in-fold selector kept each time."""
    preds, kept = [], []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=r)
        cv = cross_validate(model, X, y, cv=skf, return_estimator=True, return_indices=True, n_jobs=-1)
        oof = np.empty_like(y)
        for est, test in zip(cv["estimator"], cv["indices"]["test"]):
            oof[test] = est.predict(X[test])
            if "select" in est.named_steps:
                kept.append(est.named_steps["select"].get_support())
        preds.append(oof)
    return np.concatenate(preds), (np.array(kept) if kept else None)


def feature_subset_curve(X, y, classes, feature_names, n_repeats):
    y_pooled = np.concatenate([y] * n_repeats)
    rows, freq_rows = [], []

    for k in FEATURE_SUBSET_SIZES:
        print(f"    top-{k} features", flush=True)
        model = Pipeline(
            [
                ("scaler", MinMaxScaler()),
                ("select", SelectFromModel(_rf(200), max_features=k, threshold=-np.inf)),
                ("clf", _rf(300)),
            ]
        )
        pred, kept = _oof_with_selection(model, X, y, n_repeats)
        freq = kept.mean(axis=0)
        top = [feature_names[i] for i in np.argsort(-freq)[:k]]
        rows.append({"subset": f"top {k} (chosen in-fold)", "n_features": k, **score(y_pooled, pred, classes),
                     "features_most_often_chosen": "; ".join(top)})
        freq_rows.append({"n_features": k, **{f: 100 * v for f, v in zip(feature_names, freq)}})

    # The paper's four, taken at face value (no selection step).
    print("    paper's four key attributes", flush=True)
    idx = [feature_names.index(f) for f in PAPER_KEY_ATTRIBUTES]
    model = Pipeline([("scaler", MinMaxScaler()), ("clf", _rf(300))])
    pred, _ = _oof_with_selection(model, X[:, idx], y, n_repeats)
    rows.append({"subset": "Ma et al.'s 4 key attributes", "n_features": len(idx), **score(y_pooled, pred, classes),
                 "features_most_often_chosen": "; ".join(PAPER_KEY_ATTRIBUTES)})

    return pd.DataFrame(rows), pd.DataFrame(freq_rows)


# --------------------------------------------------------------------------
# Figures
# --------------------------------------------------------------------------
def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)


def plot_permutation(final_imp, rf_imp, final_name, figures_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
    for ax, imp, title in ((axes[0], final_imp, f"Best Review-2 model\n{final_name}"),
                           (axes[1], rf_imp, "Random Forest (Review-1 configuration)")):
        d = imp.sort_values("f1_drop_mean")
        colours = [GOLD if f in PAPER_KEY_ATTRIBUTES else BERRY for f in d["feature"]]
        ax.barh(d["feature"], d["f1_drop_mean"], xerr=d["f1_drop_std"], color=colours,
                error_kw={"ecolor": MUTED, "elinewidth": 1})
        ax.axvline(0, color=INK, lw=0.8)
        ax.set_xlabel("Macro-F1 lost when the feature is shuffled (percentage points)")
        ax.set_title(title, fontsize=10)
        _style(ax)
    # Explicit patches: legend entries from empty bar() calls lose their colour.
    axes[1].legend(handles=[Patch(color=GOLD, label="Highlighted by Ma et al. (Figs. 4-5)"),
                            Patch(color=BERRY, label="Other features")],
                   frameon=False, loc="lower right", fontsize=8)
    fig.suptitle("Permutation importance on held-out folds (error bars: spread across the 10 folds)", fontsize=11)
    fig.tight_layout()
    fig.savefig(figures_dir / "C1_permutation_importance.png", dpi=170)
    plt.close(fig)


def plot_methods_and_direction(table, figures_dir):
    d = table.sort_values("final_f1_drop", ascending=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), sharey=True)

    ax = axes[0]
    shares = {
        "Permutation, best model": d["final_f1_drop"].clip(lower=0) / d["final_f1_drop"].clip(lower=0).sum(),
        "Permutation, Random Forest": d["rf_f1_drop"].clip(lower=0) / d["rf_f1_drop"].clip(lower=0).sum(),
        "Impurity (MDI), Random Forest": d["rf_mdi"] / d["rf_mdi"].sum(),
    }
    y_pos = np.arange(len(d))
    for i, ((label, vals), colour) in enumerate(zip(shares.items(), (BERRY, ROSE, MUTED))):
        ax.barh(y_pos + (i - 1) * 0.27, 100 * vals, 0.27, color=colour, label=label)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(d["feature"])
    ax.set_xlabel("Share of total importance (%)")
    ax.set_title("Three importance measures, side by side", fontsize=10)
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    _style(ax)

    ax = axes[1]
    colours = [SAGE if r > 0 else BERRY for r in d["spearman_rho"]]
    ax.barh(y_pos, d["spearman_rho"], color=colours)
    ax.axvline(0, color=INK, lw=0.8)
    ax.set_xlim(-0.55, 0.55)
    ax.set_xlabel("Spearman correlation with quality")
    ax.set_title("Direction: higher value -> higher (green) or lower (red) quality", fontsize=10)
    _style(ax)

    fig.tight_layout()
    fig.savefig(figures_dir / "C2_importance_methods_and_direction.png", dpi=170)
    plt.close(fig)


def plot_subset_curve(curve, figures_dir):
    topk = curve[curve["subset"].str.startswith("top")]
    paper = curve[~curve["subset"].str.startswith("top")].iloc[0]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(topk["n_features"], topk["accuracy"], marker="o", color=MUTED, label="Accuracy")
    ax.plot(topk["n_features"], topk["macro_f1"], marker="o", color=BERRY, label="Macro-F1 (primary)")
    ax.scatter([paper["n_features"]], [paper["accuracy"]], marker="D", s=70, color=GOLD, zorder=5,
               label="Ma et al.'s 4 attributes (accuracy)")
    ax.scatter([paper["n_features"]], [paper["macro_f1"]], marker="D", s=70, color=GOLD, edgecolor=INK, zorder=5,
               label="Ma et al.'s 4 attributes (macro-F1)")
    for _, r in topk.iterrows():
        ax.annotate(f"{r['accuracy']:.1f}", (r["n_features"], r["accuracy"]), textcoords="offset points",
                    xytext=(0, 7), ha="center", fontsize=8)
        ax.annotate(f"{r['macro_f1']:.1f}", (r["n_features"], r["macro_f1"]), textcoords="offset points",
                    xytext=(0, -13), ha="center", fontsize=8)
    ax.set_xticks(FEATURE_SUBSET_SIZES)
    ax.set_xlabel("Number of lab measurements used (chosen inside each training fold)")
    ax.set_ylabel("%")
    ax.set_title("How many lab tests does the model need?  (Random Forest, class-weighted)")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    _style(ax)
    fig.tight_layout()
    fig.savefig(figures_dir / "C3_feature_subset.png", dpi=170)
    plt.close(fig)


# --------------------------------------------------------------------------
# Driver
# --------------------------------------------------------------------------
def run_feature_importance(X, y, classes, feature_names, final_model_name, n_repeats, tables_dir, figures_dir):
    feature_names = list(feature_names)

    print(f"  C1. permutation importance -- best model: {final_model_name}", flush=True)
    final_imp = permutation_importance_cv(estimator_for(final_model_name), X, y, feature_names)
    print("  C1. permutation importance -- Random Forest", flush=True)
    rf_imp = permutation_importance_cv(get_base_learners()["RF"], X, y, feature_names)
    direction = direction_of_effect(X, y, feature_names)

    table = pd.DataFrame({"feature": feature_names})
    table["final_f1_drop"] = final_imp["f1_drop_mean"].values
    table["final_f1_drop_std"] = final_imp["f1_drop_std"].values
    table["final_accuracy_drop"] = final_imp["accuracy_drop_mean"].values
    table["rf_f1_drop"] = rf_imp["f1_drop_mean"].values
    table["rf_f1_drop_std"] = rf_imp["f1_drop_std"].values
    table["rf_accuracy_drop"] = rf_imp["accuracy_drop_mean"].values
    table["rf_mdi"] = rf_imp["mdi_mean"].values
    table = table.merge(direction, on="feature")
    table["rank_final"] = table["final_f1_drop"].rank(ascending=False).astype(int)
    table["rank_rf_permutation"] = table["rf_f1_drop"].rank(ascending=False).astype(int)
    table["rank_rf_mdi"] = table["rf_mdi"].rank(ascending=False).astype(int)
    table["highlighted_by_paper"] = table["feature"].isin(PAPER_KEY_ATTRIBUTES)
    table = table.sort_values("rank_final")
    table.round(4).to_csv(tables_dir / "C_feature_importance.csv", index=False)

    # How well do the three rankings agree with each other?
    agree = pd.DataFrame(
        [
            ("permutation (best model) vs permutation (RF)",
             spearmanr(table["final_f1_drop"], table["rf_f1_drop"])[0]),
            ("permutation (best model) vs MDI (RF)", spearmanr(table["final_f1_drop"], table["rf_mdi"])[0]),
            ("permutation (RF) vs MDI (RF)", spearmanr(table["rf_f1_drop"], table["rf_mdi"])[0]),
        ],
        columns=["comparison", "spearman_rank_agreement"],
    )
    agree.round(4).to_csv(tables_dir / "C_ranking_agreement.csv", index=False)

    # Cross-check with Ma et al. (2024) Figs. 4-5.
    top4 = list(table.head(4)["feature"])
    cross = pd.DataFrame(
        [
            {
                "paper_attribute": f,
                "our_rank_best_model": int(table.loc[table["feature"] == f, "rank_final"].iloc[0]),
                "our_rank_rf": int(table.loc[table["feature"] == f, "rank_rf_permutation"].iloc[0]),
                "in_our_top4": f in top4,
            }
            for f in PAPER_KEY_ATTRIBUTES
        ]
    )
    cross.to_csv(tables_dir / "C_paper_crosscheck.csv", index=False)

    print("  C2. feature-subset curve", flush=True)
    curve, freq = feature_subset_curve(X, y, classes, feature_names, n_repeats)
    curve.round(4).to_csv(tables_dir / "C_feature_subset.csv", index=False)
    freq.round(2).to_csv(tables_dir / "C_selection_frequency.csv", index=False)

    plot_permutation(final_imp, rf_imp, final_model_name, figures_dir)
    plot_methods_and_direction(table, figures_dir)
    plot_subset_curve(curve, figures_dir)
    return table, agree, cross, curve

In [ ]:
%%writefile run_all.py
"""
run_all.py

Single entry point for the reproduction of Ma et al. (2024).

Usage:
    python run_all.py                    # full protocol: 10-fold CV x 10 repeats
    python run_all.py --quick            # fast sanity pass: 10-fold CV x 2 repeats
    python run_all.py --review2          # + Review 2: class balancing, improvements,
                                         #   feature importance (about 45-60 min)
    python run_all.py --quick --review2  # the same at 2 repeats, to a scratch folder
    python run_all.py --with-novelty     # also run the optional Step 4 extension

By default this runs the base-paper reproduction only. The clustering /
outlier-detection / association-rule extension is optional coursework and is
kept out of the default path -- pass --with-novelty to include it.

All tables are written to results/tables/, all figures to results/figures/.
Review 2 writes to results/review2/{tables,figures}/ so the Review-1 numbers
are never overwritten.
"""
import argparse
import sys
from pathlib import Path

import pandas as pd

SRC_DIR = Path(__file__).resolve().parent / "src"
sys.path.insert(0, str(SRC_DIR))

from config import (  # noqa: E402
    FIGURES_DIR,
    N_REPEATS,
    QUICK_N_REPEATS,
    QUICK_OUTPUT_DIR,
    QUICK_REVIEW2_DIR,
    REVIEW2_FIGURES_DIR,
    REVIEW2_TABLES_DIR,
    TABLES_DIR,
)
from ensemble import run_repeated_cv, weight_concentration  # noqa: E402
from evaluate import (  # noqa: E402
    evaluate_all,
    paper_internal_consistency_check,
    paper_table9_reconciliation,
    per_class_metrics,
)
from preprocessing import load_red_wine_xy  # noqa: E402


def step3_reproduction(X, y, n_repeats, tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR):
    print("=" * 72)
    print("STEP 3: Base paper reproduction (classical-ML substitutions)")
    print("=" * 72)

    results = run_repeated_cv(X, y, n_repeats=n_repeats)
    all_results, comparison = evaluate_all(
        results["pooled_true"],
        results["pooled_preds"],
        results["pooled_ensemble"],
        results["classes"],
        pooled_ensemble_hard=results["pooled_ensemble_hard"],
        tables_dir=tables_dir,
        figures_dir=figures_dir,
    )

    print("\n-- Model ranking (pooled over all repeats) --")
    summary = pd.read_csv(tables_dir / "model_accuracy_summary.csv")
    print(summary.to_string(index=False))

    print("\n-- Per-class comparison vs. published benchmarks --")
    print(comparison.to_string(index=False))

    print("\n-- Ensemble weights, and why they matter --")
    conc = weight_concentration(results["per_repeat"][0]["weights"])
    for key, value in conc.items():
        print(f"  {key}: {value}")
    print(
        "  (Weights close to uniform => the ensemble approximates a plain average\n"
        "   of its members, which lands between the strongest and weakest learner.)"
    )

    print("\n-- Reconciling the paper's reported 89.73% against its own Table 9 --")
    recon, per_class_check = paper_table9_reconciliation()
    print(recon.to_string(index=False))
    print("\n  Per-class check (does their 'precision' reproduce column-wise?):")
    print(per_class_check.to_string(index=False))
    print(
        "\n  CAVEAT: Table 9 was transcribed by eye from the printed PDF and may\n"
        "  contain reading errors. Reported as 'could not reconcile', not as a\n"
        "  claim that the paper is incorrect."
    )

    print("\n-- Paper internal consistency: Table 8 weights x Tables 4-7 accuracies --")
    ens_acc = 100.0 * (results["pooled_ensemble"] == results["pooled_true"]).mean()
    print(paper_internal_consistency_check(our_ensemble_accuracy=ens_acc).to_string(index=False))

    return results, all_results, comparison


def step4_novelty(X, y, df, baseline_all_results, n_repeats):
    """Optional extension -- only runs with --with-novelty."""
    from config import FEATURE_COLUMNS
    from novelty_cluster_mine_feedback import (
        augment_with_cluster_feature,
        compare_cluster_feedback,
        crosscheck_with_paper_figures,
        run_clustering,
        run_frequent_pattern_mining,
        run_outlier_detection,
        save_cluster_feedback_outputs,
        save_clustering_outputs,
        save_mining_outputs,
        save_outlier_outputs,
    )
    from preprocessing import minmax_normalize

    print("\n" + "=" * 72)
    print("STEP 4 (optional): Clustering, outliers, cluster feedback, rule mining")
    print("=" * 72)

    # The unsupervised analyses fit on all rows at once (no train/test split),
    # so scaling the whole matrix here introduces no leakage.
    X_norm, _ = minmax_normalize(X)

    print("\n-- 4a. Clustering --")
    clustering_result = run_clustering(X_norm)
    crosstab = save_clustering_outputs(clustering_result, y)
    print(f"Best clustering: {clustering_result['best_method']} (k={clustering_result['best_k']})")
    print(crosstab)

    print("\n-- 4b. Outlier detection --")
    _, rate_by_class = run_outlier_detection(X_norm, y)
    save_outlier_outputs(rate_by_class)
    print(rate_by_class)

    print("\n-- 4c. Cluster-ID feedback feature --")
    X_aug = augment_with_cluster_feature(X, clustering_result["best_labels"], clustering_result["best_k"])
    aug_results = run_repeated_cv(X_aug, y, n_repeats=n_repeats)
    aug_metrics = per_class_metrics(
        aug_results["pooled_true"], aug_results["pooled_ensemble"], aug_results["classes"]
    )
    feedback_comparison = compare_cluster_feedback(baseline_all_results["Ensemble"][1], aug_metrics)
    save_cluster_feedback_outputs(feedback_comparison)
    print(feedback_comparison.to_string(index=False))

    print("\n-- 4d. Frequent pattern mining --")
    top10, per_cluster = run_frequent_pattern_mining(df, FEATURE_COLUMNS, y, clustering_result["best_labels"])
    crosscheck = crosscheck_with_paper_figures(top10)
    save_mining_outputs(top10, per_cluster, crosscheck)
    print("\nCross-check vs. Ma et al. (2024) Figs. 4-5 key attributes:")
    for key, value in crosscheck.items():
        print(f"  {key}: {sorted(value)}")


def step5_review2(X, y, feature_columns, n_repeats, tables_dir, figures_dir):
    """Review 2: (A) class balancing, (B) improving the result, (C) feature
    importance. Writes to results/review2/ only -- see src/review2.py."""
    import numpy as np

    from config import BENCHMARK_CLASSES, N_REPEATS, REVIEW2_PRIMARY_METRIC
    from feature_importance import run_feature_importance
    from review2 import (
        BALANCING_LABELS,
        PAPER_ENSEMBLE,
        build_model_comparison,
        experiment_a_class_balancing,
        experiment_b1_temperature,
        experiment_b2_b3_improvements,
        headline_table,
        plot_headline,
        paired_comparison,
        per_repeat_scores,
        plot_tradeoff,
        save_confusion,
    )

    print("\n" + "=" * 72)
    print("REVIEW 2: class balancing, improving the result, feature importance")
    print(f"Primary metric, fixed before any Review-2 run: {REVIEW2_PRIMARY_METRIC}")
    print("=" * 72)
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    classes = sorted(np.unique(y).tolist())
    y_pooled = np.concatenate([y] * n_repeats)
    show = ["accuracy", "balanced_accuracy", "macro_f1"] + [f"recall_{c}" for c in BENCHMARK_CLASSES]

    print("\n-- A. Class balancing (same four learners, paper's weighted soft vote) --")
    a_df, cache, a_preds = experiment_a_class_balancing(X, y, classes, n_repeats, tables_dir, figures_dir)
    ens = a_df[a_df["model"] == PAPER_ENSEMBLE]
    print(ens[["balancing_label"] + show].round(2).to_string(index=False))

    # The "partial" scheme IS the Review-1 configuration, so on the full
    # protocol it must reproduce the Review-1 ensemble to the last digit.
    if n_repeats == N_REPEATS:
        canon = pd.read_csv(TABLES_DIR / "model_accuracy_summary.csv")
        expected = canon.loc[canon["model"] == "Ensemble (soft vote)", "overall_accuracy_pct"].iloc[0]
        got = ens.loc[ens["balancing"] == "partial", "accuracy"].iloc[0]
        status = "MATCH" if abs(got - expected) < 1e-9 else "MISMATCH -- investigate before trusting Review 2"
        print(f"\n  Regression check vs Review 1: {got:.10f} vs {expected:.10f} -> {status}")

    best_balancing = ens.sort_values(REVIEW2_PRIMARY_METRIC, ascending=False)["balancing"].iloc[0]
    print(f"\n  Best balancing scheme on {REVIEW2_PRIMARY_METRIC}: {BALANCING_LABELS[best_balancing]}")

    print("\n-- B1. Weight sharpening (diagnostic test of the Review-1 explanation) --")
    configs = ["partial"] + ([best_balancing] if best_balancing != "partial" else [])
    b1 = experiment_b1_temperature(cache, y, classes, configs, tables_dir, figures_dir)
    print(b1[["balancing", "sharpen", "temperature", "accuracy", "macro_f1", "rf_accuracy",
              "largest_weight_learner", "largest_weight"]].round(3).to_string(index=False))

    print("\n-- B2/B3. Stacking and stronger learners --")
    b_df, b_preds = experiment_b2_b3_improvements(X, y, classes, n_repeats, cache, tables_dir)
    print(b_df[["model"] + show].round(2).to_string(index=False))

    comp = build_model_comparison(a_df, b_df, tables_dir)
    head, best_name = headline_table(a_df, comp, tables_dir)
    plot_headline(comp, figures_dir)
    plot_tradeoff(comp, figures_dir)
    all_preds = {**a_preds, **b_preds}
    save_confusion(y_pooled, all_preds[best_name], classes, "review2_best", tables_dir)
    save_confusion(y_pooled, a_preds[f"{PAPER_ENSEMBLE} | {BALANCING_LABELS['partial']}"], classes,
                   "review1_ensemble", tables_dir)

    # Keep every candidate's pooled predictions, so any later question can be
    # answered without re-running the whole experiment.
    np.savez_compressed(tables_dir / "pooled_predictions.npz", y_pooled=y_pooled,
                        names=np.array(list(all_preds)), preds=np.array(list(all_preds.values())))

    # Is the improvement consistent across the 10 repeats, or noise?
    r1_ens = f"{PAPER_ENSEMBLE} | {BALANCING_LABELS['partial']}"
    r1_rf = f"RF | {BALANCING_LABELS['partial']}"
    lt_r1 = f"Learned-temperature ensemble | {BALANCING_LABELS['partial']}"
    best_acc_name = comp.sort_values("accuracy", ascending=False)["model"].iloc[0]
    per_rep = per_repeat_scores(all_preds, y, classes, n_repeats)
    per_rep.round(4).to_csv(tables_dir / "per_repeat_scores.csv", index=False)
    runner_up = comp["model"].iloc[1]
    paired = paired_comparison(per_rep, [(best_name, r1_ens), (best_name, r1_rf), (best_name, runner_up),
                                         (lt_r1, r1_ens), (lt_r1, r1_rf), (best_acc_name, r1_ens)], tables_dir)
    print("\n-- Paired comparison across the repeats (wins / Wilcoxon p; p is optimistic, see code) --")
    print(paired.round(4).to_string(index=False))

    print(f"\n-- Every candidate, ranked by {REVIEW2_PRIMARY_METRIC} (top 10) --")
    print(comp[["model", "accuracy", "balanced_accuracy", "macro_f1"]].head(10).round(2).to_string(index=False))
    best_acc = comp.sort_values("accuracy", ascending=False).iloc[0]
    print(f"\n  Best on {REVIEW2_PRIMARY_METRIC}: {best_name}")
    print(f"  Best on accuracy:  {best_acc['model']} ({best_acc['accuracy']:.2f}%)")
    print("\n-- Review 1 vs Review 2 --")
    print(head.round(2).to_string(index=False))

    print("\n-- C. Feature importance --")
    table, agree, cross, curve = run_feature_importance(
        X, y, classes, feature_columns, best_name, n_repeats, tables_dir, figures_dir
    )
    print(table[["feature", "final_f1_drop", "rf_f1_drop", "rf_mdi", "spearman_rho", "rank_final"]]
          .round(3).to_string(index=False))
    print("\n  Ranking agreement between methods:")
    print(agree.round(3).to_string(index=False))
    print("\n  Ma et al.'s four key attributes in our ranking:")
    print(cross.to_string(index=False))
    print("\n  How many lab tests are needed:")
    print(curve[["subset", "accuracy", "macro_f1", "features_most_often_chosen"]].round(2).to_string(index=False))
    return comp, head, table


def main():
    parser = argparse.ArgumentParser(description="Reproduce Ma et al. (2024) on the UCI red wine dataset.")
    parser.add_argument(
        "--quick",
        action="store_true",
        help=f"use {QUICK_N_REPEATS} CV repeats instead of {N_REPEATS} for a fast sanity pass",
    )
    parser.add_argument(
        "--with-novelty",
        action="store_true",
        help="also run the optional Step 4 clustering / outlier / rule-mining extension",
    )
    parser.add_argument(
        "--review2",
        action="store_true",
        help="also run Review 2: class balancing, improvements and feature importance",
    )
    args = parser.parse_args()

    n_repeats = QUICK_N_REPEATS if args.quick else N_REPEATS

    # --quick runs a reduced protocol, so its output must NOT overwrite the
    # canonical 10-repeat results that README.md and report/methodology.md
    # quote. Send it to a scratch directory instead -- the same protection the
    # walkthrough notebook already uses.
    if args.quick:
        tables_dir = QUICK_OUTPUT_DIR
        figures_dir = QUICK_OUTPUT_DIR
        tables_dir.mkdir(parents=True, exist_ok=True)
    else:
        tables_dir, figures_dir = TABLES_DIR, FIGURES_DIR

    X, y, feature_columns, df = load_red_wine_xy()
    print(f"Loaded red wine dataset: {df.shape[0]} rows, {len(feature_columns)} features.")
    print(f"Protocol: 10-fold stratified CV x {n_repeats} repeats "
          f"({10 * n_repeats} total fold-runs).")
    if args.quick:
        print(f"QUICK MODE: reduced protocol -- output goes to {tables_dir}, "
              f"NOT to results/. The canonical results are left untouched.")
    print()

    _, baseline_all_results, _ = step3_reproduction(
        X, y, n_repeats, tables_dir=tables_dir, figures_dir=figures_dir
    )

    if args.with_novelty:
        if args.quick:
            print("\n(Skipping Step 4: the extension writes to results/ and would "
                  "overwrite canonical output. Run --with-novelty without --quick.)")
        else:
            step4_novelty(X, y, df, baseline_all_results, n_repeats)

    if args.review2:
        if args.quick:
            r2_tables, r2_figures = QUICK_REVIEW2_DIR / "tables", QUICK_REVIEW2_DIR / "figures"
        else:
            r2_tables, r2_figures = REVIEW2_TABLES_DIR, REVIEW2_FIGURES_DIR
        step5_review2(X, y, feature_columns, n_repeats, r2_tables, r2_figures)

    print("\n" + "=" * 72)
    if args.quick:
        print(f"DONE (quick mode). Output -> {tables_dir}")
    else:
        print("DONE. Tables -> results/tables/ , figures -> results/figures/")
    if args.review2:
        print("Review 2 -> " + str(QUICK_REVIEW2_DIR if args.quick else REVIEW2_TABLES_DIR.parent))
    else:
        print("(Review 2 skipped; pass --review2 to include it.)")
    if not args.with_novelty:
        print("(Step 4 extension skipped; pass --with-novelty to include it.)")
    print("=" * 72)


if __name__ == "__main__":
    main()

## 5. Run the reproduction

`--quick` uses 2 CV repeats instead of 10. On a 2-core Colab instance the full
10-repeat protocol takes several minutes, so the quick pass is the sensible default
for a live demo — it exercises exactly the same code path.

**To run the full canonical protocol** (the numbers on the slides), change the flag in
the cell below to `python run_all.py` and expect roughly 5–10 minutes on Colab.

In [ ]:
import sys
# `!python` would resolve via PATH, which is not necessarily the interpreter this
# notebook is running on. `sys.executable` is. (This bit me during testing: a shell
# escape picked up a different install with an older scikit-learn and produced
# slightly different numbers.)
!"{sys.executable}" run_all.py --quick

## 6. Results

`evaluate.py` uses Matplotlib's headless `Agg` backend (it writes PNGs rather than
drawing to a screen), so the figure is loaded back and displayed here.

In [ ]:
from IPython.display import Image, display

# --quick writes to results/_quick_run/ so a reduced run can never be mistaken for
# the canonical 10-repeat output. Fall back to results/ for a full run.
OUT = Path("results/_quick_run")
if not (OUT / "model_accuracy_summary.csv").exists():
    OUT = Path("results/tables")
FIG = Path("results/_quick_run/accuracy_comparison.png")
if not FIG.exists():
    FIG = Path("results/figures/accuracy_comparison.png")

print("Reading results from:", OUT)
display(Image(filename=str(FIG)))

In [ ]:
import pandas as pd

print("=" * 64)
print("MODEL RANKING   (slide 12)")
print("=" * 64)
display(pd.read_csv(OUT / "model_accuracy_summary.csv").round(2))

print("=" * 64)
print("PER-CLASS vs PUBLISHED BENCHMARKS   (slide 13)")
print("=" * 64)
display(pd.read_csv(OUT / "benchmark_comparison.csv").round(2))

### The central finding — slide 14

Both tables below are computed **from the paper's own printed tables**, not from our
model. They are the evidence behind the three checks on slide 14.

In [ ]:
print("=" * 64)
print("CHECKS 1 AND 2 -- reconciling the paper's Table 9 against its own text")
print("=" * 64)
display(pd.read_csv(OUT / "paper_table9_reconciliation.csv"))

print("=" * 64)
print("CHECK 3 -- the paper's Table 8 weights applied to its own base accuracies")
print("=" * 64)
display(pd.read_csv(OUT / "paper_internal_consistency_check.csv"))

> **Caveat, stated on slide 14 and repeated here:** the Table 9 cells were transcribed by
> eye from the printed PDF and may contain reading errors. This is reported as *"we could
> not reconcile these figures"*, **not** as a claim that the paper is incorrect.

---

## 7. Optional — Review 2: class balancing, improvements, feature importance

This reruns the Review-2 experiments at the reduced 2-repeat protocol. It is
**slow on Colab's free tier — budget roughly an hour** (it fits several hundred
models, some of them with nested cross-validation). The full 10-repeat results
quoted on the Review-2 slides come from `python run_all.py --review2` on a laptop.

It needs `imbalanced-learn` (for SMOTE), which Colab does not ship.

In [ ]:
import sys
!"{sys.executable}" -m pip install -q imbalanced-learn
import imblearn, sklearn
print("imbalanced-learn", imblearn.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
import sys
!"{sys.executable}" run_all.py --quick --review2

In [ ]:
from IPython.display import Image, display
R2 = Path("results/_quick_run/review2/figures")
for name in ["A_class_balancing", "B1_weight_temperature", "B_tradeoff", "B4_chosen_temperatures",
             "C1_permutation_importance", "C2_importance_methods_and_direction", "C3_feature_subset"]:
    f = R2 / f"{name}.png"
    if f.exists():
        print(name); display(Image(filename=str(f)))
display(pd.read_csv("results/_quick_run/review2/tables/headline_review1_vs_review2.csv").round(2))

---

## 8. Optional — the unsupervised extension

**Skip this unless a panel member asks about it.** This is the appendix work behind backup
slide 19 (clustering, outlier detection, cluster-ID feedback, and FP-Growth association
rules). It is not part of the reproduction and is kept behind a flag for that reason.

It needs one package Colab does not ship, `mlxtend`, so it takes an extra moment to set up.

In [ ]:
import sys
# NOTE: mlxtend depends on scikit-learn, so pip may UPGRADE scikit-learn in this
# environment as a side effect. Harmless in a throwaway Colab runtime; be aware of
# it if you ever run this notebook against a local install you care about.
!"{sys.executable}" -m pip install -q mlxtend
print("mlxtend installed. Check the scikit-learn version above still applies.")

In [ ]:
%%writefile src/novelty_cluster_mine_feedback.py
"""
novelty_cluster_mine_feedback.py

Novelty extension targeting Ma et al. (2024)'s own stated limitation: poor
precision on the extreme quality classes (3, 4, 8), which the paper attributes
to small sample size. Four parts:

  4a. Unsupervised clustering (K-means + Ward's linkage agglomerative) on the
      11 physicochemical features only (no quality label). k = 3..8, best k
      chosen by silhouette score. Cross-tabulated against actual quality
      class to see whether clusters found from attributes alone line up with
      quality tiers.
  4b. Outlier detection (Isolation Forest + Local Outlier Factor). Checks
      whether classes 3/4/8 are flagged as outliers at a higher rate than the
      well-represented classes 5/6/7 -- i.e. whether "small sample" also means
      "genuinely atypical," not just "few."
  4c. Feed the best clustering's cluster ID back into the Step-3 ensemble as a
      12th input feature; compare per-class precision with vs. without it,
      especially for classes 3, 4, 8.
  4d. Frequent pattern mining (fpgrowth + association_rules) on a
      Low(3,4)/High(7,8) quality-tier split (Medium 5,6 dropped for cleaner,
      higher-lift rules), run on the whole dataset and separately within each
      cluster from 4a. Rules are filtered to consequent = "quality=High",
      sorted by lift, and cross-checked against the attributes Ma et al.
      (2024) Figs. 4-5 flag as most important (alcohol, sulphates, volatile
      acidity, citric acid).
"""
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from mlxtend.frequent_patterns import association_rules, fpgrowth
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.neighbors import LocalOutlierFactor

RESULTS_DIR = Path(__file__).resolve().parent.parent / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

RANDOM_STATE = 42
K_RANGE = range(3, 9)
CONTAMINATION = 0.10
FIG4_5_KEY_ATTRIBUTES = {"alcohol", "sulphates", "volatile acidity", "citric acid"}


# ---------------------------------------------------------------------------
# 4a. Clustering
# ---------------------------------------------------------------------------
def run_clustering(X, k_range=K_RANGE, random_state=RANDOM_STATE):
    """K-means and Ward agglomerative clustering over k_range; silhouette
    score for each; returns the best (method, k, labels) by silhouette."""
    rows = []
    labels_by_method_k = {}

    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        km_labels = km.fit_predict(X)
        rows.append({"method": "kmeans", "k": k, "silhouette": silhouette_score(X, km_labels)})
        labels_by_method_k[("kmeans", k)] = km_labels

        ward = AgglomerativeClustering(n_clusters=k, linkage="ward")
        ward_labels = ward.fit_predict(X)
        rows.append({"method": "ward", "k": k, "silhouette": silhouette_score(X, ward_labels)})
        labels_by_method_k[("ward", k)] = ward_labels

    silhouette_df = pd.DataFrame(rows)
    best_row = silhouette_df.loc[silhouette_df["silhouette"].idxmax()]
    best_method, best_k = best_row["method"], int(best_row["k"])
    best_labels = labels_by_method_k[(best_method, best_k)]

    return {
        "silhouette_df": silhouette_df,
        "best_method": best_method,
        "best_k": best_k,
        "best_labels": best_labels,
        "labels_by_method_k": labels_by_method_k,
    }


def cluster_quality_crosstab(cluster_labels, y):
    return pd.crosstab(
        pd.Series(cluster_labels, name="cluster"), pd.Series(y, name="quality_class")
    )


def save_clustering_outputs(clustering_result, y, tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR):
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    clustering_result["silhouette_df"].to_csv(tables_dir / "clustering_silhouette_scores.csv", index=False)

    crosstab = cluster_quality_crosstab(clustering_result["best_labels"], y)
    crosstab.to_csv(tables_dir / "cluster_quality_crosstab.csv")

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(crosstab, annot=True, fmt="d", cmap="viridis", ax=ax)
    ax.set_title(
        f"Cluster vs. quality class "
        f"(best: {clustering_result['best_method']}, k={clustering_result['best_k']})"
    )
    fig.tight_layout()
    fig.savefig(figures_dir / "cluster_quality_heatmap.png", dpi=150)
    plt.close(fig)

    return crosstab


# ---------------------------------------------------------------------------
# 4b. Outlier detection
# ---------------------------------------------------------------------------
def run_outlier_detection(X, y, contamination=CONTAMINATION, random_state=RANDOM_STATE):
    iso = IsolationForest(contamination=contamination, random_state=random_state)
    iso_outlier = iso.fit_predict(X) == -1

    lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
    lof_outlier = lof.fit_predict(X) == -1

    per_sample = pd.DataFrame(
        {"quality_class": y, "isolation_forest_outlier": iso_outlier, "lof_outlier": lof_outlier}
    )
    rate_by_class = (
        per_sample.groupby("quality_class")[["isolation_forest_outlier", "lof_outlier"]].mean() * 100
    )
    rate_by_class.columns = ["isolation_forest_outlier_pct", "lof_outlier_pct"]
    return per_sample, rate_by_class


def save_outlier_outputs(rate_by_class, tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR):
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    rate_by_class.to_csv(tables_dir / "outlier_rates_by_class.csv")

    fig, ax = plt.subplots(figsize=(8, 5))
    rate_by_class.plot(kind="bar", ax=ax)
    ax.set_xlabel("Quality class")
    ax.set_ylabel("Outlier rate (%)")
    ax.set_title("Outlier rate by quality class (extreme classes 3/4/8 vs. 5/6/7)")
    ax.legend(["Isolation Forest", "Local Outlier Factor"])
    fig.tight_layout()
    fig.savefig(figures_dir / "outlier_rate_by_class.png", dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 4c. Feedback into the classifier
# ---------------------------------------------------------------------------
def augment_with_cluster_feature(X, cluster_labels, n_clusters):
    """Append cluster id as a 12th feature, min-max scaled to [0, 1] to match
    the scale of the other (already min-max normalized) features."""
    denom = max(n_clusters - 1, 1)
    cluster_feature = (np.asarray(cluster_labels, dtype=float) / denom).reshape(-1, 1)
    return np.hstack([X, cluster_feature])


def compare_cluster_feedback(metrics_without, metrics_with, classes_of_interest=(3, 4, 8)):
    """Compare per-class "precision" (paper definition = recall) with vs.
    without the cluster-ID feature, for the given classes of interest."""
    without = metrics_without.set_index("class")["precision_paper_definition_pct"]
    with_ = metrics_with.set_index("class")["precision_paper_definition_pct"]
    all_classes = sorted(set(without.index) | set(with_.index))

    comparison = pd.DataFrame(
        {
            "class": all_classes,
            "precision_without_cluster_feature_pct": [without.get(c, np.nan) for c in all_classes],
            "precision_with_cluster_feature_pct": [with_.get(c, np.nan) for c in all_classes],
        }
    )
    comparison["delta_pct"] = (
        comparison["precision_with_cluster_feature_pct"] - comparison["precision_without_cluster_feature_pct"]
    )
    comparison["is_extreme_class_of_interest"] = comparison["class"].isin(classes_of_interest)
    return comparison


def save_cluster_feedback_outputs(comparison_df, tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR):
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(tables_dir / "cluster_feedback_precision_comparison.csv", index=False)

    labels = [str(c) for c in comparison_df["class"]]
    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width / 2, comparison_df["precision_without_cluster_feature_pct"], width, label="Without cluster ID")
    ax.bar(x + width / 2, comparison_df["precision_with_cluster_feature_pct"], width, label="With cluster ID")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Quality class")
    ax.set_ylabel("Precision (paper definition = recall) (%)")
    ax.set_title("Ensemble per-class precision: before vs. after cluster-ID feedback feature")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figures_dir / "cluster_feedback_precision_comparison.png", dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 4d. Frequent pattern mining
# ---------------------------------------------------------------------------
def discretize_features(raw_feature_df, feature_columns, n_bins=3, bin_labels=("Low", "Med", "High")):
    """Discretize each continuous feature into `n_bins` quantile bins."""
    discretized = pd.DataFrame(index=raw_feature_df.index)
    for col in feature_columns:
        try:
            binned = pd.qcut(raw_feature_df[col], q=n_bins, labels=bin_labels[:n_bins], duplicates="drop")
        except ValueError:
            # Too many repeated values for `n_bins` distinct quantile edges;
            # fall back to ranking first so every row still gets a bin.
            binned = pd.qcut(raw_feature_df[col].rank(method="first"), q=n_bins, labels=bin_labels[:n_bins])
        discretized[col] = binned.astype(str)
    return discretized


def compute_quality_tier(y):
    """Low = classes 3-4, High = classes 7-8, Medium (5-6) is dropped later
    for cleaner, higher-lift rules -- both dropped classes are the
    best-represented and least in need of extra scrutiny here."""
    tier = np.full(len(y), "Medium", dtype=object)
    tier[np.isin(y, [3, 4])] = "Low"
    tier[np.isin(y, [7, 8])] = "High"
    return tier


def _one_hot_items(discretized_df):
    items = pd.DataFrame(index=discretized_df.index)
    for col in discretized_df.columns:
        dummies = pd.get_dummies(discretized_df[col], prefix=col, prefix_sep="=")
        items = pd.concat([items, dummies], axis=1)
    return items


def build_basket(discretized_df, tier, row_mask):
    sub_disc = discretized_df.loc[row_mask]
    sub_tier = pd.Series(np.asarray(tier)[row_mask], index=sub_disc.index)
    items = _one_hot_items(sub_disc)
    tier_dummies = pd.get_dummies(sub_tier, prefix="quality", prefix_sep="=")
    basket = pd.concat([items, tier_dummies], axis=1).fillna(False).astype(bool)
    return basket


def mine_rules(basket, min_support=0.05):
    if basket.shape[0] == 0:
        return pd.DataFrame()
    freq_items = fpgrowth(basket, min_support=min_support, use_colnames=True)
    if freq_items.empty:
        return pd.DataFrame()
    return association_rules(freq_items, num_itemsets=len(basket), metric="lift", min_threshold=1.0)


def filter_high_quality_rules(rules, top_n=10):
    if rules.empty:
        return rules
    mask = rules["consequents"].apply(lambda s: set(s) == {"quality=High"})
    return rules.loc[mask].sort_values("lift", ascending=False).head(top_n)


def rule_antecedent_attributes(rules_df):
    attrs = set()
    for antecedents in rules_df.get("antecedents", []):
        for item in antecedents:
            attrs.add(item.split("=")[0])
    return attrs


def run_frequent_pattern_mining(
    raw_feature_df, feature_columns, y, cluster_labels, n_bins=3, min_support=0.05, min_cluster_rows=10
):
    discretized_df = discretize_features(raw_feature_df, feature_columns, n_bins)
    tier = compute_quality_tier(y)
    low_high_mask = tier != "Medium"

    basket_full = build_basket(discretized_df, tier, low_high_mask)
    rules_full = mine_rules(basket_full, min_support=min_support)
    top10_overall = filter_high_quality_rules(rules_full, top_n=10)

    per_cluster_top5 = {}
    for cluster_id in sorted(np.unique(cluster_labels)):
        cluster_mask = low_high_mask & (np.asarray(cluster_labels) == cluster_id)
        if cluster_mask.sum() < min_cluster_rows:
            per_cluster_top5[cluster_id] = pd.DataFrame()
            continue
        basket_c = build_basket(discretized_df, tier, cluster_mask)
        rules_c = mine_rules(basket_c, min_support=min_support)
        per_cluster_top5[cluster_id] = filter_high_quality_rules(rules_c, top_n=5)

    return top10_overall, per_cluster_top5


def crosscheck_with_paper_figures(top10_overall):
    mined_attrs = rule_antecedent_attributes(top10_overall)
    agree = mined_attrs & FIG4_5_KEY_ATTRIBUTES
    disagree_mined_only = mined_attrs - FIG4_5_KEY_ATTRIBUTES
    disagree_paper_only = FIG4_5_KEY_ATTRIBUTES - mined_attrs
    return {
        "mined_attributes": mined_attrs,
        "paper_key_attributes": FIG4_5_KEY_ATTRIBUTES,
        "agreement": agree,
        "mined_only": disagree_mined_only,
        "paper_only": disagree_paper_only,
    }


def _rules_to_readable(rules_df):
    if rules_df.empty:
        return rules_df
    out = rules_df.copy()
    out["antecedents"] = out["antecedents"].apply(lambda s: ", ".join(sorted(s)))
    out["consequents"] = out["consequents"].apply(lambda s: ", ".join(sorted(s)))
    keep_cols = ["antecedents", "consequents", "support", "confidence", "lift"]
    return out[keep_cols]


def save_mining_outputs(top10_overall, per_cluster_top5, crosscheck, tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR):
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    readable_overall = _rules_to_readable(top10_overall)
    readable_overall.to_csv(tables_dir / "top_association_rules_overall.csv", index=False)

    per_cluster_frames = []
    for cluster_id, rules_df in per_cluster_top5.items():
        readable = _rules_to_readable(rules_df)
        readable = readable.copy()
        readable.insert(0, "cluster", cluster_id)
        per_cluster_frames.append(readable)
    per_cluster_combined = (
        pd.concat(per_cluster_frames, ignore_index=True) if per_cluster_frames else pd.DataFrame()
    )
    per_cluster_combined.to_csv(tables_dir / "top_association_rules_per_cluster.csv", index=False)

    # Markdown + image renderings of the overall top-10 table (deliverable
    # asks for the rules "as image or markdown table").
    md_path = figures_dir / "top_association_rules.md"
    with open(md_path, "w") as f:
        f.write("# Top 10 association rules for quality=High (overall, sorted by lift)\n\n")
        f.write(readable_overall.round(4).to_markdown(index=False) if not readable_overall.empty else "_No rules found at this support threshold._\n")
        f.write("\n\n## Cross-check against Ma et al. (2024) Figs. 4-5 key attributes\n\n")
        f.write(f"- Attributes appearing in our mined rules: {sorted(crosscheck['mined_attributes'])}\n")
        f.write(f"- Paper's flagged key attributes (alcohol, sulphates, volatile acidity, citric acid): {sorted(crosscheck['paper_key_attributes'])}\n")
        f.write(f"- Agreement: {sorted(crosscheck['agreement'])}\n")
        f.write(f"- In our rules only: {sorted(crosscheck['mined_only'])}\n")
        f.write(f"- In paper's figures only, not in our top rules: {sorted(crosscheck['paper_only'])}\n")

    if not readable_overall.empty:
        fig, ax = plt.subplots(figsize=(12, 0.5 + 0.4 * len(readable_overall)))
        ax.axis("off")
        table_data = readable_overall.round(3).values
        tbl = ax.table(
            cellText=table_data,
            colLabels=readable_overall.columns,
            loc="center",
            cellLoc="left",
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8)
        tbl.auto_set_column_width(list(range(len(readable_overall.columns))))
        ax.set_title("Top 10 association rules for quality=High (overall, by lift)", pad=20)
        fig.tight_layout()
        fig.savefig(figures_dir / "top_association_rules.png", dpi=150, bbox_inches="tight")
        plt.close(fig)


if __name__ == "__main__":
    from evaluate import per_class_metrics
    from ensemble import run_repeated_cv
    from preprocessing import FEATURE_COLUMNS, load_normalized_red_wine

    X, y, feature_columns, df = load_normalized_red_wine()

    print("Running clustering (4a)...")
    clustering_result = run_clustering(X)
    print(f"Best clustering: {clustering_result['best_method']} k={clustering_result['best_k']}")
    save_clustering_outputs(clustering_result, y)

    print("Running outlier detection (4b)...")
    _, rate_by_class = run_outlier_detection(X, y)
    save_outlier_outputs(rate_by_class)
    print(rate_by_class)

    print("Running cluster-feedback ensemble comparison (4c, quick 2-repeat smoke test)...")
    baseline_results = run_repeated_cv(X, y, n_repeats=2)
    baseline_metrics = per_class_metrics(
        baseline_results["pooled_true"], baseline_results["pooled_ensemble"], baseline_results["classes"]
    )
    X_aug = augment_with_cluster_feature(X, clustering_result["best_labels"], clustering_result["best_k"])
    aug_results = run_repeated_cv(X_aug, y, n_repeats=2)
    aug_metrics = per_class_metrics(
        aug_results["pooled_true"], aug_results["pooled_ensemble"], aug_results["classes"]
    )
    comparison = compare_cluster_feedback(baseline_metrics, aug_metrics)
    save_cluster_feedback_outputs(comparison)
    print(comparison)

    print("Running frequent pattern mining (4d)...")
    top10, per_cluster = run_frequent_pattern_mining(df, FEATURE_COLUMNS, y, clustering_result["best_labels"])
    crosscheck = crosscheck_with_paper_figures(top10)
    save_mining_outputs(top10, per_cluster, crosscheck)
    print(crosscheck)

In [ ]:
import sys
# Runs the full 10-repeat protocol plus the extension -- several minutes on Colab.
!"{sys.executable}" run_all.py --with-novelty

---

### Notes

- Every number printed above is also written to a CSV under `results/`. Nothing in the
  slides was typed by hand.
- The random seed is fixed at 42 in `src/config.py`, so a re-run reproduces the same
  numbers on the same package versions.
- Colab resets its filesystem when the runtime disconnects. That is fine — `Run all`
  rebuilds everything from scratch in a couple of minutes.

**References**

1. Ma, R., Mao, D., Cao, D., Luo, S., Gupta, S., & Wang, Y. (2024). From vineyard to table:
   Uncovering wine quality for sales management through machine learning.
   *Journal of Business Research*, 176, 114576.
2. Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). Modeling wine
   preferences by data mining from physicochemical properties.
   *Decision Support Systems*, 47(4), 547–553.
3. UCI Machine Learning Repository — Wine Quality Dataset.
4. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.